<a href="https://colab.research.google.com/github/Velgarath/recomendador_juegos_bgg/blob/main/juegosBGG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **PASO 0. IMPORTACION DE LIBRERIAS**

In [44]:
# @title
# LIBRERIAS
import re
import requests
import pandas as pd
import numpy as np
import xml.etree.ElementTree as ET
import time
import os
import shutil
from math import prod
from google.colab import userdata
from google.colab import drive
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import HuberRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor



#**PASO 1: MONTAMOS GOOGLE DRIVE EN COLAB PARA RECUPERAR INFORMACION**

In [2]:
# @title
#MONTAMOS EL DRIVE DE GOOGLE
from google.colab import drive

# 1. Intentamos desmontar de forma limpia usando la herramienta de disco de Colab
try:
    drive.flush_and_unmount()
    print("Unidad desmontada limpiamente.")
except Exception as e:
    print("No se pudo desmontar de forma estándar (puede que no estuviera montado):", e)

# 2. Si quedan carpetas residuales en la ruta, las eliminamos a la fuerza
mountpoint = '/content/drive'
if os.path.exists(mountpoint):
    try:
        # Eliminamos la carpeta "drive" y todo su contenido conflictivo local
        shutil.rmtree(mountpoint)
        print(f"Carpeta residual '{mountpoint}' eliminada con éxito.")
    except Exception as e:
        print(f"No se pudo eliminar la carpeta {mountpoint} directamente: {e}")
        print("Intentando forzar la eliminación por consola...")
        # Alternativa por comandos de sistema si Python encuentra el directorio bloqueado
        !umount -l /content/drive
        !rm -rf /content/drive

# 3. Volvemos a montar desde cero de forma limpia
print("Volviendo a montar Drive...")
drive.mount(mountpoint)

Drive not mounted, so nothing to flush and unmount.
Unidad desmontada limpiamente.
Volviendo a montar Drive...
Mounted at /content/drive


# **PASO 2: RENOVAMOS LOS FICHEROS KAGLE DE INTERNET (OPCIONAL)**

In [ ]:
# @title
#ESTE CODIGO ME RENUEVA LOS FICHEROS DE KAGGLE DE LA BGG Y ME LOS METE EN GOOGLE DRIVE
#Importacion de la variable de entorno para conectarme a KAGGLE y descargar los datos
try:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
    print("¡Tokens de Kaggle cargados de forma segura desde los secretos de Colab!")
except Exception as e:
    print("Error:",e)

#Descarga de los datos de la web de Kaggle
!kaggle datasets download -d threnjen/board-games-database-from-boardgamegeek -p "/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/" --unzip


¡Tokens de Kaggle cargados de forma segura desde los secretos de Colab!
Dataset URL: https://www.kaggle.com/datasets/threnjen/board-games-database-from-boardgamegeek
License(s): CC-BY-SA-3.0
board-games-database-from-boardgamegeek.zip: Skipping, found more recently modified local copy (use --force to force download)


# **PASO 3: LECTURA DE FICHEROS CSV DE KAGLE**

In [3]:
# @title
#LECTURA DE LOS DATOS CSV DE KAGGLE
#Leemos los ficheros de la base de datos de Kagle y los metemos en DataFrames

# Tabla principal de juegos
df_games = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/games.csv")
# Tabla de mecánicas
df_mechanics = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/mechanics.csv")
# Tabla de temáticas
df_themes = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/themes.csv")
# Tabla de categorías
df_subcategories = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/subcategories.csv")
# Tabla de diseñadores
df_designers = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/designers_reduced.csv")
# Tabla de editoriales
df_publishers = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/publishers_reduced.csv")




# **PASO 4: LECTURA DE LOS FICHEROS QUE CONTIENEN LOS XML DE JUEGOS DE MESA QUE TENGO Y QUE HE EVALUADO**

In [4]:
# @title
# LECTURA DE LOS DATOS DE LA BGG
# Este link descarga el XML de los juegos que poseo. NO inlcuye expansiones
# https://boardgamegeek.com/xmlapi2/collection?username=velgarath&subtype=boardgame&excludesubtype=boardgameexpansion&own=1
# Este link descarga el XML de los juegos que he puntuado en la BGG independientemente de si los tengo o no los tengo. No incluye expansiones.
# https://boardgamegeek.com/xmlapi2/collection?username=velgarath&subtype=boardgame&excludesubtype=boardgameexpansion&rated=1


#Esta funcion es necesaria para limpiar los ampersands del XML
#Es un problema conocido de la exportacion de los XML de la BGG
def limpia_ampersands(texto_xml):
    # Reemplaza & que NO sea parte de una entidad válida (&amp; &lt; &gt; &quot; &apos; &#123; &#x1F;)
    return re.sub(r'&(?!amp;|lt;|gt;|quot;|apos;|#\d+;|#x[0-9a-fA-F]+;)', '&amp;', texto_xml)

#drive.mount('/content/drive', force_remount=True)

# Abrimos por separado los juegos poseidos por luis y los juegos puntuiados por luis
f = open("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/velgarath_games_owned.xml", "r")
juegos_owned_luis = limpia_ampersands(f.read())
f.close()

f = open("/content/drive/MyDrive/Colab Notebooks/IA Juegos/Datos/velgarath_games_rated.xml", "r")
juegos_rated_luis = limpia_ampersands(f.read())
f.close()





# **PASO 5: PARSEAMOS LOS XML DE LA BGG Y CREACION DE DATA FRAMES "OWNED" Y "RATED"**



*   Este procedimiento es importante porque aqui tenemos que asegurarnos leer todas las columnas de features que queremos añadir al data frame


In [5]:
# @title
# PARSEADO XML DE UNA COLECCION DE JUEGOS DE MESA IMPORTADO DE LA BGG

def convierte_xml_a_dataframe(respuesta):
    raiz = ET.fromstring(respuesta)
    lista_juegos = []

    for item in raiz.findall('item'):
        objectid = item.attrib.get('objectid')  # clave para el merge con Kaggle
        subtype = item.attrib.get('subtype')

        name = item.find('name').text if item.find('name') is not None else 'Desconocido'
        year = item.find('yearpublished').text if item.find('yearpublished') is not None else 'N/A'
        numplays = item.find('numplays').text if item.find('numplays') is not None else 'N/A'

        status = item.find('status')
        own = status.attrib.get('own') if status is not None else None
        wishlist = status.attrib.get('wishlist') if status is not None else None
        preordered = status.attrib.get('preordered') if status is not None else None

        # stats puede faltar según cómo se exportó el XML, así que protegemos todo
        stats = item.find('stats')
        if stats is not None:
            min_players = stats.attrib.get('minplayers')
            max_players = stats.attrib.get('maxplayers')
            playing_time = stats.attrib.get('playingtime')
            num_owned = stats.attrib.get('numowned')
            etiqueta_rating = stats.find('rating')
            user_rating = etiqueta_rating.attrib.get('value') if etiqueta_rating is not None else 'N/A'
            average_elem = stats.find('rating/average')
            avg_rating = average_elem.attrib.get('value') if average_elem is not None else None
        else:
            min_players = max_players = playing_time = num_owned = None
            user_rating = 'N/A'
            avg_rating = None

        lista_juegos.append({
            'BGGId': objectid,
            'Subtype': subtype,
            'IsExpansion': subtype == 'boardgameexpansion',
            'Name': name,
            'YearPublished': year,
            'AvgRating': avg_rating,
            'UserRating': user_rating,
            'MinPlayers': min_players,
            'MaxPlayers': max_players,
            'PlayingTime': playing_time,
            'NumOwned': num_owned,
            'NumPlays': numplays,
            'Own': own,
            'Wishlist': wishlist,
            'Preordered': preordered,
        })

    df_lista_juegos = pd.DataFrame(lista_juegos)

    columns_to_convert = ['YearPublished', 'AvgRating', 'UserRating',
                          'MinPlayers', 'MaxPlayers',
                          'PlayingTime', 'NumOwned', 'NumPlays']
    df_lista_juegos[columns_to_convert] = df_lista_juegos[columns_to_convert].apply(pd.to_numeric, errors='coerce')
    df_lista_juegos['BGGId'] = pd.to_numeric(df_lista_juegos['BGGId'], errors='coerce').astype('Int64')

    return df_lista_juegos

# Creamos 2 dataframes
# el data frame RATED lo usaremos para entrenar el modelo.
# el data frame OWNED no lo usaremos en la fase de aprendizaje. Lo usaremos para ver qué juegos de los que recomienda el modelo YA lo tenemos.
df_rated = convierte_xml_a_dataframe(juegos_rated_luis)
df_owned = convierte_xml_a_dataframe(juegos_owned_luis)

#Hemos visto que el tipo BGG_ID es "Int64". Lo pasamos a "int64 para que sea igual de la BD de Kagle"
df_owned['BGGId'] = df_owned['BGGId'].astype('int64')
df_rated['BGGId'] = df_rated['BGGId'].astype('int64')

print("Ratings válidos en OWNED:", df_owned['UserRating'].notna().sum())
print("Ratings válidos en RATED:", df_rated['UserRating'].notna().sum())




Ratings válidos en OWNED: 222
Ratings válidos en RATED: 279


# **PASO 6: PREPARACION DE LOS DATA FRAMES DE ENTRENAMIENTO**

Esta parte es IMPORTANTE. Hacemos el **MERGE** de los data frames de mi coleccion de la BGG con los data frames de Kagle, añadiendo características.

*   Construimos dataframes progresivamente más "anchos" añadiendo nuevas familias
*   Hemos hecho "bins" de Year Published
*   Hemos hecho one-hot encoding de alguna característica.




In [6]:
# @title
# ==============================================================
# PREPARACIÓN DE LOS DATAFRAMES DE ENTRENAMIENTO
# ==============================================================

cols_luis = ['BGGId', 'UserRating']
# ==============================================================
# TRANSFORMACIÓN DE YearPublished
# ==============================================================

bins = [
    -float('inf'),
    1979,
    1989,
    1999,
    2009,
    2019,
    float('inf')
]

labels = [
    'Pre1980',
    '1980s',
    '1990s',
    '2000s',
    '2010s',
    '2020s'
]

df_games['YearPeriod'] = pd.cut(
    df_games['YearPublished'],
    bins=bins,
    labels=labels
)

df_year = pd.get_dummies(
    df_games[['BGGId', 'YearPeriod']],
    columns=['YearPeriod'],
    dtype=int
)


# ==============================================================
# ONE-HOT ENCODING DE Family
# ==============================================================

df_family = pd.get_dummies(
    df_games[['BGGId', 'Family']],
    columns=['Family'],
    dummy_na=False,
    dtype=int
)

# ==============================================================
# DATAFRAME GLOBAL PREPARADO
# ==============================================================

df_games_encoded = df_games.merge(
    df_year,
    on='BGGId',
    how='left'
)

df_games_encoded = df_games_encoded.merge(
    df_family,
    on='BGGId',
    how='left'
)

# ==============================================================
# DATAFRAMES GLOBALES PROGRESIVAMENTE MÁS ANCHOS
# ==============================================================

df_games_mechanics = df_games_encoded.merge(
    df_mechanics,
    on='BGGId',
    how='left'
)

df_games_themes = df_games_mechanics.merge(
    df_themes,
    on='BGGId',
    how='left'
)

df_games_subcategories = df_games_themes.merge(
    df_subcategories,
    on='BGGId',
    how='left'
)

df_games_designers = df_games_subcategories.merge(
    df_designers,
    on='BGGId',
    how='left'
)

df_games_publishers = df_games_designers.merge(
    df_publishers,
    on='BGGId',
    how='left'
)

# ==============================================================
# DATAFRAME BASE DE ENTRENAMIENTO
# ==============================================================

df_train = df_rated[cols_luis].merge(
    df_games_encoded,
    on='BGGId',
    how='inner'
)


# Sustituimos valores ausentes por la mediana de la columna.
df_train['ComAgeRec'] = df_train['ComAgeRec'].fillna(
    df_train['ComAgeRec'].median()
)

df_train['LanguageEase'] = df_train['LanguageEase'].fillna(
    df_train['LanguageEase'].median()
)

# ==============================================================
# DATAFRAMES DE ENTRENAMIENTO PROGRESIVAMENTE MÁS ANCHOS
# ==============================================================

df_train_mechanics = df_train.merge(
    df_mechanics,
    on='BGGId',
    how='left'
)

df_train_themes = df_train_mechanics.merge(
    df_themes,
    on='BGGId',
    how='left'
)

df_train_subcategories = df_train_themes.merge(
    df_subcategories,
    on='BGGId',
    how='left'
)

df_train_designers = df_train_subcategories.merge(
    df_designers,
    on='BGGId',
    how='left'
)

df_train_publishers = df_train_designers.merge(
    df_publishers,
    on='BGGId',
    how='left'
)




# **PASO 7: FUNCIONES DE MACHINE LEARNING QUE APLICAREMOS**

In [30]:
# Necesitamos eliminar ciertos caracteres de las cabeceras de los dataframes para que XBoost funcione
def prepara_X_para_modelo(X, nombre_modelo):
    # Creamos una copia para no modificar
    # el dataframe original.
    X_preparado = X.copy()
    # ============================================================
    # XGBOOST
    # ============================================================
    if nombre_modelo == 'XGBoost':

        X_preparado.columns = [
            str(columna)
                .replace('[', '(')
                .replace(']', ')')
                .replace('<', ' menor que ')
            for columna in X_preparado.columns
        ]
    # ============================================================
    # LIGHTGBM
    # ============================================================
    elif nombre_modelo == 'LightGBM':
        nombres_columnas = []
        for indice, columna in enumerate(X_preparado.columns):
            nombre_limpio = re.sub(
                r'[^A-Za-z0-9_]+',
                '_',
                str(columna)
            )
            nombre_limpio = (
                'feature_'
                + str(indice)
                + '_'
                + nombre_limpio
            )
            nombres_columnas.append(
                nombre_limpio
            )
        X_preparado.columns = nombres_columnas

    # ============================================================
    # CATBOOST
    # ============================================================
    elif nombre_modelo == 'CatBoost':
        # De momento no hacemos nada.
        #
        # Si más adelante CatBoost necesita alguna
        # preparación específica, la añadiremos aquí.
        pass

    # ============================================================
    # RESTO DE MODELOS
    # ============================================================

    # Linear, Ridge, Lasso, ElasticNet,
    # RandomForest, ExtraTrees,
    # GradientBoosting, etc.
    #
    # No necesitan ninguna transformación especial.
    return X_preparado

## 7.1. Modelo Dummy. Hacemos la media de todas mis puntuaciones. Es el "objetivo a batir"

In [7]:
# @title
# ==============================================================
# MODELO DUMMY
# Baseline muy simple: siempre predice la nota media.
# Sirve como referencia mínima para saber si nuestros modelos
# realmente están aprendiendo algo útil.
# ==============================================================

def evalua_dummy(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # DummyRegressor no intenta aprender relaciones entre las
    # features de X y la respuesta y.
    #
    # Con strategy='mean' siempre predice la media de y calculada
    # sobre los datos de entrenamiento.
    #
    # Ejemplo:
    # si la nota media del entrenamiento es 7.0,
    # el modelo predice 7.0 para todos los juegos.
    #
    # Esto nos da un baseline:
    # cualquier modelo serio debería intentar mejorar este resultado.

    modelo = DummyRegressor(strategy='mean')


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # cross_val_score divide los datos en 5 pliegues (folds).
    #
    # En cada iteración:
    #   - entrena con 4 folds
    #   - valida con el fold restante
    #
    # Aunque el modelo Dummy sea muy simple, usamos exactamente
    # el mismo sistema de validación que para el resto de modelos.
    #
    # scoring='neg_mean_absolute_error'
    # indica que queremos evaluar usando MAE.
    #
    # Scikit-learn devuelve el MAE en negativo por convención,
    # porque internamente considera que "mayor score = mejor".

    scores = cross_val_score(
        modelo,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------
    #
    # Cambiamos el signo para convertir los valores negativos
    # de scikit-learn en errores MAE positivos.
    #
    # Luego calculamos la media de los 5 folds.

    mae = -scores.mean()


    # ----------------------------------------------------------
    # 4. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # Devolvemos:
    #
    # mae
    #   -> MAE medio de los 5 folds.
    #
    # -scores
    #   -> MAE individual de cada fold, convertido a positivo.
    #
    # No devolvemos coeficientes porque el Dummy no aprende
    # relaciones entre features.
    #
    # Tampoco necesitamos entrenarlo con todos los datos para
    # inspeccionarlo, porque su comportamiento es trivial:
    # siempre predice la media.

    return mae, -scores

## 7.2. Regresión Lineal

In [8]:
# @title
# ==============================================================
# REGRESIÓN LINEAL
# Busca una relación lineal entre las features X y la respuesta y.
#
# La predicción tiene aproximadamente esta forma:
#
# y_predicha =
#     intercepto
#     + peso_1 * feature_1
#     + peso_2 * feature_2
#     + ...
#
# A diferencia de Ridge, aquí NO existe regularización.
# ==============================================================

def evalua_LinearRegression(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # LinearRegression intenta encontrar los coeficientes
    # que mejor ajustan una relación lineal entre X e y.
    #
    # No aplica ninguna penalización a los coeficientes.
    #
    # Esto hace que sea muy interpretable, pero también puede
    # hacerlo más sensible a:
    #   - outliers
    #   - multicolinealidad
    #   - demasiadas features
    #   - overfitting

    modelo_linear = LinearRegression()


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Dividimos los datos en 5 folds.
    #
    # En cada iteración:
    #   - entrenamos con 4 folds
    #   - validamos con el fold restante
    #
    # Así obtenemos una estimación más robusta del rendimiento
    # que si usáramos una única división train/test.
    #
    # scoring='neg_mean_absolute_error'
    # pide a scikit-learn que evalúe usando MAE.
    #
    # Como scikit-learn devuelve el resultado en negativo,
    # luego cambiaremos el signo.

    scores_linear = cross_val_score(
        modelo_linear,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------
    #
    # Convertimos los scores a MAE positivos
    # y calculamos la media de los 5 folds.

    mae_linear = -scores_linear.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # La validación cruzada sirve para evaluar el rendimiento.
    #
    # Después entrenamos nuevamente el modelo con TODO X e y
    # para disponer de un modelo final que podamos:
    #   - inspeccionar
    #   - comparar con otros modelos
    #   - utilizar posteriormente para hacer predicciones

    modelo_linear.fit(X, y)


    # ----------------------------------------------------------
    # 5. EXTRAEMOS LOS COEFICIENTES APRENDIDOS
    # ----------------------------------------------------------
    #
    # .coef_ contiene los pesos que LinearRegression ha aprendido
    # para cada feature.
    #
    # Creamos una Series de pandas para asociar cada coeficiente
    # con el nombre de su variable.
    #
    # sort_values() los ordena de menor a mayor.
    #
    # IMPORTANTE:
    # aquí las variables NO están estandarizadas.
    #
    # Por tanto, el tamaño de los coeficientes NO se puede comparar
    # directamente entre features que tengan escalas muy distintas.
    #
    # Ejemplo:
    # un coeficiente pequeño en MfgPlaytime puede seguir teniendo
    # mucho efecto si esa variable toma valores muy grandes.

    pesos_linear = pd.Series(
        modelo_linear.coef_,
        index=X.columns
    ).sort_values()


    # ----------------------------------------------------------
    # 6. EXTRAEMOS EL INTERCEPTO
    # ----------------------------------------------------------
    #
    # El intercepto es el término constante de la ecuación lineal.
    #
    # Es el valor base al que luego se suman o restan
    # las contribuciones de las distintas features.

    intercepto_linear = modelo_linear.intercept_


    # ----------------------------------------------------------
    # 7. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # Devolvemos:
    #
    # modelo_linear
    #   -> modelo entrenado con todos los datos.
    #
    # mae_linear
    #   -> MAE medio de los 5 folds.
    #
    # -scores_linear
    #   -> MAE individual de cada fold, convertido a positivo.
    #
    # pesos_linear
    #   -> coeficientes aprendidos por la regresión lineal.
    #
    # intercepto_linear
    #   -> término constante de la ecuación.

    return (
        modelo_linear,
        mae_linear,
        -scores_linear,
        pesos_linear,
        intercepto_linear
    )


## 7.3. Regresión Ridge

In [9]:
# @title
# ==============================================================
# REGRESIÓN RIDGE
# Regresión lineal con regularización L2.
#
# Ridge intenta reducir el overfitting penalizando coeficientes
# excesivamente grandes.
#
# La predicción sigue siendo una ecuación lineal:
#
# y_predicha =
#     intercepto
#     + peso_1 * feature_1
#     + peso_2 * feature_2
#     + ...
#
# La diferencia frente a LinearRegression es que Ridge introduce
# una penalización que intenta mantener controlados los coeficientes.
# ==============================================================

def evalua_Ridge(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # make_pipeline encadena varios pasos que se ejecutan
    # automáticamente en orden.
    #
    # En nuestro caso:
    #
    #   1º StandardScaler()
    #      Estandariza todas las variables.
    #
    #      Después de escalar, cada feature continua tiene
    #      aproximadamente:
    #
    #          media = 0
    #          desviación estándar = 1
    #
    #      Esto es especialmente importante para Ridge porque
    #      queremos que la penalización de los coeficientes sea
    #      comparable entre variables con escalas diferentes.
    #
    #   2º Ridge(alpha=1.0)
    #      Aplica la regresión Ridge sobre las variables escaladas.
    #
    # alpha controla la fuerza de la regularización:
    #
    #   alpha pequeño -> Ridge se parece más a LinearRegression
    #   alpha grande  -> mayor penalización de los coeficientes
    #
    # Por ahora usamos alpha=1.0 sin intentar optimizarlo.

    modelo_ridge = make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Dividimos nuestros datos en 5 folds.
    #
    # En cada iteración:
    #
    #   - entrenamos con 4 folds
    #   - validamos con el fold restante
    #
    # IMPORTANTE:
    #
    # Como StandardScaler está dentro del pipeline,
    # en cada fold calcula la media y desviación estándar
    # exclusivamente usando los datos de entrenamiento.
    #
    # De esta manera evitamos que información del fold de
    # validación se utilice accidentalmente durante el escalado.

    scores_ridge = cross_val_score(
        modelo_ridge,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------
    #
    # Scikit-learn devuelve el MAE en negativo por convención.
    #
    # Cambiamos el signo y calculamos la media de los cinco folds.

    mae_ridge = -scores_ridge.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # La validación cruzada nos sirve para evaluar cómo generaliza.
    #
    # Después entrenamos nuevamente utilizando TODO X e y.
    #
    # Así obtenemos un modelo final que podemos:
    #
    #   - inspeccionar
    #   - utilizar para realizar predicciones
    #   - comparar con otros algoritmos

    modelo_ridge.fit(X, y)


    # ----------------------------------------------------------
    # 5. EXTRAEMOS LOS COEFICIENTES APRENDIDOS
    # ----------------------------------------------------------
    #
    # modelo_ridge es un pipeline formado por:
    #
    #   StandardScaler
    #   Ridge
    #
    # Con:
    #
    # modelo_ridge.named_steps['ridge']
    #
    # accedemos específicamente al modelo Ridge.
    #
    # .coef_ contiene el peso aprendido para cada feature.
    #
    # Como las variables han sido estandarizadas, estos
    # coeficientes son mucho más comparables entre sí que
    # los coeficientes de LinearRegression sin escalado.
    #
    # Un coeficiente:
    #
    #   positivo -> asociación con ratings más altos
    #   negativo -> asociación con ratings más bajos
    #
    # Esto NO implica causalidad.

    pesos_ridge = pd.Series(
        modelo_ridge.named_steps['ridge'].coef_,
        index=X.columns
    ).sort_values()


    # ----------------------------------------------------------
    # 6. EXTRAEMOS EL INTERCEPTO
    # ----------------------------------------------------------
    #
    # .intercept_ contiene el término constante de la ecuación.
    #
    # Podemos imaginar la predicción como:
    #
    # rating =
    #     intercepto
    #     + efectos positivos de algunas features
    #     - efectos negativos de otras features
    #
    # Como StandardScaler centra las variables alrededor de 0,
    # el intercepto de Ridge se aproxima a la predicción cuando
    # las features están en sus valores medios.
    #
    # Por eso suele tener una interpretación más intuitiva que
    # el intercepto de una regresión sin estandarizar.

    intercepto_ridge = modelo_ridge.named_steps['ridge'].intercept_


    # ----------------------------------------------------------
    # 7. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # modelo_ridge
    #   -> modelo ya entrenado con todos los datos.
    #
    # mae_ridge
    #   -> MAE medio obtenido mediante validación cruzada.
    #
    # -scores_ridge
    #   -> MAE de cada uno de los cinco folds.
    #
    # pesos_ridge
    #   -> coeficientes aprendidos para cada feature.
    #
    # intercepto_ridge
    #   -> término constante de la ecuación lineal.

    return (
        modelo_ridge,
        mae_ridge,
        -scores_ridge,
        pesos_ridge,
        intercepto_ridge
    )

## 7.4. Regresión Huber Regressor

In [10]:
# @title
# ==============================================================
# HUBER REGRESSOR
# Regresión lineal robusta frente a observaciones extremas.
#
# Para errores normales se comporta parecido a una regresión
# cuadrática convencional.
#
# Para errores muy grandes reduce su influencia, evitando que
# unos pocos outliers dominen todo el ajuste.
# ==============================================================

def evalua_Huber(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # Usamos StandardScaler porque Huber funciona mejor cuando
    # las variables están en escalas comparables.
    #
    # epsilon=1.35 es el valor por defecto.
    # Controla a partir de qué nivel un residuo empieza a tratarse
    # como "extremo".

    #modelo_huber = make_pipeline(
    #    StandardScaler(),
    #    HuberRegressor(epsilon=1.35)
    #)

    modelo = make_pipeline(
    StandardScaler(),
    HuberRegressor(
        epsilon=1.35,
        max_iter=5000
    )
)


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------

    scores_huber = cross_val_score(
        modelo_huber,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------

    mae_huber = -scores_huber.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS CON TODOS LOS DATOS
    # ----------------------------------------------------------

    modelo_huber.fit(X, y)


    # ----------------------------------------------------------
    # 5. EXTRAEMOS LOS COEFICIENTES
    # ----------------------------------------------------------
    #
    # Igual que en Ridge, accedemos al modelo que está dentro
    # del pipeline.

    pesos_huber = pd.Series(
        modelo_huber.named_steps['huberregressor'].coef_,
        index=X.columns
    ).sort_values()


    # ----------------------------------------------------------
    # 6. EXTRAEMOS EL INTERCEPTO
    # ----------------------------------------------------------

    intercepto_huber = modelo_huber.named_steps['huberregressor'].intercept_


    # ----------------------------------------------------------
    # 7. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------

    return (
        modelo_huber,
        mae_huber,
        -scores_huber,
        pesos_huber,
        intercepto_huber
    )

## 7.5. Regresión Lasso

In [11]:
# @title
# ==============================================================
# REGRESIÓN LASSO
# Regresión lineal con regularización L1.
#
# Lasso penaliza el tamaño de los coeficientes y puede hacer que
# algunos de ellos lleguen exactamente a 0.
#
# Esto significa que, además de regularizar el modelo,
# puede realizar una especie de selección automática de features.
# ==============================================================

def evalua_Lasso(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # Utilizamos StandardScaler porque la regularización Lasso
    # depende del tamaño de los coeficientes.
    #
    # Si una variable tiene una escala enorme y otra una escala
    # muy pequeña, la penalización no sería comparable.
    #
    # StandardScaler transforma las variables para que tengan
    # aproximadamente:
    #
    #   media = 0
    #   desviación estándar = 1
    #
    # alpha controla la intensidad de la regularización.
    #
    # alpha pequeño:
    #   Lasso se parece más a una regresión lineal normal.
    #
    # alpha grande:
    #   mayor penalización y más coeficientes pueden quedar en 0.
    #
    # Por ahora usamos alpha=0.1 como punto de partida.

    modelo_lasso = make_pipeline(
        StandardScaler(),
        Lasso(alpha=0.1)
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Igual que con los otros modelos:
    #
    #   - dividimos los datos en 5 folds
    #   - entrenamos con 4
    #   - validamos con el restante
    #
    # El scaler está dentro del pipeline, por lo que se calcula
    # únicamente con los datos de entrenamiento de cada fold.

    scores_lasso = cross_val_score(
        modelo_lasso,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------

    mae_lasso = -scores_lasso.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # Después de evaluar mediante cross-validation,
    # entrenamos el modelo con todo X e y.
    #
    # Esto nos permitirá inspeccionar los coeficientes finales.

    modelo_lasso.fit(X, y)


    # ----------------------------------------------------------
    # 5. EXTRAEMOS LOS COEFICIENTES
    # ----------------------------------------------------------
    #
    # Accedemos al modelo Lasso que está dentro del pipeline.
    #
    # .coef_ contiene el peso aprendido para cada feature.
    #
    # La característica especial de Lasso es que algunos de estos
    # coeficientes pueden ser exactamente 0.
    #
    # Si un coeficiente es 0, esa feature no participa en la
    # predicción final del modelo.

    pesos_lasso = pd.Series(
        modelo_lasso.named_steps['lasso'].coef_,
        index=X.columns
    ).sort_values()


    # ----------------------------------------------------------
    # 6. EXTRAEMOS EL INTERCEPTO
    # ----------------------------------------------------------
    #
    # Como las variables están estandarizadas, el intercepto
    # representa aproximadamente la predicción base cuando las
    # features están alrededor de sus valores medios.

    intercepto_lasso = modelo_lasso.named_steps['lasso'].intercept_


    # ----------------------------------------------------------
    # 7. CONTAMOS CUÁNTAS FEATURES HA ELIMINADO LASSO
    # ----------------------------------------------------------
    #
    # Una de las propiedades más interesantes de Lasso es que
    # puede convertir algunos coeficientes exactamente en 0.
    #
    # Contamos cuántas features ha descartado de esta manera.

    features_eliminadas = (pesos_lasso == 0).sum()


    # ----------------------------------------------------------
    # 8. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------

    return (
        modelo_lasso,
        mae_lasso,
        -scores_lasso,
        pesos_lasso,
        intercepto_lasso,
        features_eliminadas
    )

## 7.6. Regresión Elastic Net

In [12]:
# @title
# ==============================================================
# ELASTIC NET
# Regresión lineal con una combinación de regularización L1 y L2.
#
# ElasticNet mezcla las ideas de:
#
#   Ridge  -> regularización L2
#   Lasso  -> regularización L1
#
# Esto permite:
#
#   - reducir coeficientes
#   - eliminar algunas features poniéndolas a 0
#
# Puede ser útil cuando existen muchas variables relacionadas
# o correlacionadas entre sí.
# ==============================================================

def evalua_ElasticNet(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # Como ElasticNet utiliza regularización, escalamos primero
    # las variables con StandardScaler.
    #
    # alpha controla la fuerza TOTAL de la regularización.
    #
    #   alpha pequeño -> menos regularización
    #   alpha grande  -> más regularización
    #
    # l1_ratio controla cómo se reparte esa regularización entre:
    #
    #   L1 -> comportamiento parecido a Lasso
    #   L2 -> comportamiento parecido a Ridge
    #
    # Con:
    #
    #   l1_ratio = 1.0  -> prácticamente Lasso
    #   l1_ratio = 0.0  -> prácticamente Ridge
    #   l1_ratio = 0.5  -> mezcla equilibrada L1 / L2
    #
    # Para nuestro primer experimento usamos:
    #
    #   alpha = 0.1
    #   l1_ratio = 0.5
    #
    # No estamos optimizando todavía los hiperparámetros.
    # Queremos observar el comportamiento del algoritmo.

    modelo_elasticnet = make_pipeline(
        StandardScaler(),
        ElasticNet(
            alpha=0.1,
            l1_ratio=0.5
        )
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Dividimos los datos en 5 folds.
    #
    # En cada iteración:
    #
    #   - entrenamos con 4 folds
    #   - validamos con el fold restante
    #
    # Como StandardScaler está dentro del pipeline,
    # el escalado se calcula solamente con los datos
    # de entrenamiento de cada fold.

    scores_elasticnet = cross_val_score(
        modelo_elasticnet,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------

    mae_elasticnet = -scores_elasticnet.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # Después de evaluar el modelo mediante cross-validation,
    # entrenamos nuevamente utilizando todo X e y.
    #
    # Esto nos permitirá inspeccionar el modelo final.

    modelo_elasticnet.fit(X, y)


    # ----------------------------------------------------------
    # 5. EXTRAEMOS LOS COEFICIENTES
    # ----------------------------------------------------------
    #
    # Accedemos al modelo ElasticNet que está dentro del pipeline.
    #
    # Algunos coeficientes pueden quedar exactamente a 0,
    # igual que ocurre con Lasso.
    #
    # Otros simplemente pueden quedar reducidos,
    # como ocurre con Ridge.

    pesos_elasticnet = pd.Series(
        modelo_elasticnet.named_steps['elasticnet'].coef_,
        index=X.columns
    ).sort_values()


    # ----------------------------------------------------------
    # 6. EXTRAEMOS EL INTERCEPTO
    # ----------------------------------------------------------

    intercepto_elasticnet = (
        modelo_elasticnet.named_steps['elasticnet'].intercept_
    )


    # ----------------------------------------------------------
    # 7. CONTAMOS LAS FEATURES ELIMINADAS
    # ----------------------------------------------------------
    #
    # Contamos cuántas variables han terminado con
    # coeficiente exactamente igual a 0.

    features_eliminadas = (pesos_elasticnet == 0).sum()


    # ----------------------------------------------------------
    # 8. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------

    return (
        modelo_elasticnet,
        mae_elasticnet,
        -scores_elasticnet,
        pesos_elasticnet,
        intercepto_elasticnet,
        features_eliminadas
    )

## 7.7. Decision Tree

In [13]:
# @title
# ==============================================================
# DECISION TREE REGRESSOR
# Árbol de decisión para problemas de regresión.
#
# A diferencia de los modelos lineales, un árbol no construye
# una ecuación basada en coeficientes.
#
# Construye una sucesión de decisiones del tipo:
#
#     GameWeight <= 2.8 ?
#     Cat:War <= 0.5 ?
#     MaxPlayers <= 4 ?
#
# Cada división intenta crear grupos de juegos cuyas notas
# sean cada vez más parecidas entre sí.
#
# Esto permite capturar:
#
#   - relaciones no lineales
#   - interacciones entre features
#   - comportamientos diferentes según distintos rangos
# ==============================================================

def evalua_DecisionTree(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # No utilizamos StandardScaler.
    #
    # Los árboles no dependen de la escala numérica de las
    # variables porque trabajan mediante puntos de corte.
    #
    # Por ejemplo:
    #
    #     GameWeight <= 2.7
    #
    # Por ahora dejamos el árbol prácticamente sin restricciones.
    #
    # Esto es deliberado:
    # queremos observar cómo se comporta un árbol "libre"
    # antes de empezar a controlar su complejidad.
    #
    # Más adelante podremos experimentar con parámetros como:
    #
    #     max_depth
    #     min_samples_split
    #     min_samples_leaf
    #
    # random_state=42 permite que nuestros resultados sean
    # reproducibles.

    modelo_tree = DecisionTreeRegressor(
        random_state=42
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Utilizamos exactamente los mismos 5 folds que con
    # los modelos anteriores.
    #
    # Esto es fundamental para que la comparación sea justa:
    #
    #     misma X
    #     mismo y
    #     mismo sistema de evaluación
    #
    # Lo único que cambia es el algoritmo.

    scores_tree = cross_val_score(
        modelo_tree,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO
    # ----------------------------------------------------------
    #
    # Scikit-learn devuelve el MAE en negativo.
    #
    # Cambiamos el signo para obtener nuestros errores
    # habituales en positivo.

    mae_tree = -scores_tree.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # Después de evaluar mediante cross-validation,
    # entrenamos nuevamente el árbol utilizando todo X e y.
    #
    # Esto nos permitirá inspeccionar el árbol final
    # y analizar qué features ha utilizado.

    modelo_tree.fit(X, y)


    # ----------------------------------------------------------
    # 5. EXTRAEMOS LA IMPORTANCIA DE LAS FEATURES
    # ----------------------------------------------------------
    #
    # Los árboles NO tienen coeficientes como:
    #
    #     Linear
    #     Ridge
    #     Lasso
    #     ElasticNet
    #
    # En su lugar tienen:
    #
    #     feature_importances_
    #
    # Esta medida indica cuánto ha contribuido cada feature
    # a las divisiones realizadas por el árbol.
    #
    # Una importancia alta significa que esa variable ha sido
    # utilizada de forma importante para separar juegos con
    # diferentes ratings.
    #
    # Una importancia de 0 significa que el árbol no ha
    # utilizado esa feature.
    #
    # IMPORTANTE:
    # "importancia" NO significa causalidad.

    importancia_tree = pd.Series(
        modelo_tree.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)


    # ----------------------------------------------------------
    # 6. OBTENEMOS ALGUNAS CARACTERÍSTICAS DEL ÁRBOL
    # ----------------------------------------------------------
    #
    # get_depth()
    #     nos dice cuántos niveles tiene el árbol.
    #
    # get_n_leaves()
    #     nos dice cuántas hojas finales ha creado.
    #
    # Estos valores nos serán especialmente útiles para detectar
    # si el árbol se ha vuelto extremadamente complejo.

    profundidad_tree = modelo_tree.get_depth()

    hojas_tree = modelo_tree.get_n_leaves()

    # ----------------------------------------------------------
    # MEDIMOS TAMBIÉN EL ERROR SOBRE LOS DATOS DE ENTRENAMIENTO
    # ----------------------------------------------------------
    #
    # Esto NO sirve para medir la capacidad real de generalización.
    # Para eso utilizamos Cross Validation.
    #
    # Nos sirve para comparar:
    #
    #     error en entrenamiento
    #             VS
    #     error en validación
    #
    # Si el error de entrenamiento es muy pequeño pero el error
    # de validación es mucho mayor, tenemos una señal clara
    # de overfitting.

    predicciones_train_tree = modelo_tree.predict(X)

    mae_train_tree = mean_absolute_error(y,predicciones_train_tree)


    # ----------------------------------------------------------
    # 7. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # modelo_tree
    #     -> árbol entrenado con todos los datos.
    #
    # mae_tree
    #     -> MAE medio de los cinco folds.
    #
    # -scores_tree
    #     -> MAE individual de cada fold.
    #
    # importancia_tree
    #     -> importancia asignada a cada feature.
    #
    # profundidad_tree
    #     -> profundidad final del árbol.
    #
    # hojas_tree
    #     -> número de hojas creadas.
    # mae_train_tree
    #     ->

    return (
        modelo_tree,
        mae_tree,
        -scores_tree,
        importancia_tree,
        profundidad_tree,
        hojas_tree,
        mae_train_tree
    )

## 7.8 Random Forest

In [16]:
# @title
# ==============================================================
# RANDOM FOREST REGRESSOR
# Conjunto de muchos árboles de decisión.
#
# Cada árbol aprende una versión ligeramente distinta del problema
# y la predicción final es el promedio de todos ellos.
#
# La idea principal es reducir la enorme varianza que puede tener
# un árbol de decisión individual.
#
# Un árbol puede sobreajustar mucho.
# Muchos árboles distintos, combinados, suelen generalizar mejor.
# ==============================================================

def evalua_RandomForest(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # n_estimators indica cuántos árboles tendrá el bosque.
    #
    # Usamos 200 árboles como punto de partida.
    #
    # No estamos intentando optimizar todavía este parámetro.
    # Simplemente queremos observar el comportamiento natural
    # de Random Forest frente al Decision Tree individual.
    #
    # random_state=42 permite reproducir exactamente el resultado.
    #
    # n_jobs=-1 permite utilizar todos los núcleos disponibles
    # del procesador para entrenar los árboles más rápido.
    #
    # No utilizamos StandardScaler:
    # los árboles no necesitan que las variables estén escaladas.

    modelo_forest = RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Utilizamos los mismos 5 folds que para todos los modelos
    # anteriores.
    #
    # Así mantenemos una comparación justa:
    #
    #     misma X
    #     mismo y
    #     mismos folds
    #     mismo MAE
    #
    # Lo único que cambia es el algoritmo.

    scores_forest = cross_val_score(
        modelo_forest,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO DE CROSS VALIDATION
    # ----------------------------------------------------------

    mae_forest = -scores_forest.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL BOSQUE CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # Después de medir el rendimiento mediante Cross Validation,
    # entrenamos nuevamente utilizando todo X e y.
    #
    # Así tendremos un modelo final que podremos inspeccionar.

    modelo_forest.fit(X, y)


    # ----------------------------------------------------------
    # 5. CALCULAMOS EL MAE SOBRE LOS DATOS DE ENTRENAMIENTO
    # ----------------------------------------------------------
    #
    # Igual que hicimos con Decision Tree.
    #
    # Esto nos permitirá comparar:
    #
    #     MAE entrenamiento
    #             VS
    #     MAE Cross Validation
    #
    # Si entrenamiento es muchísimo mejor que CV,
    # tendremos una señal de overfitting.
    #
    # Esperamos que Random Forest siga ajustando bastante bien
    # entrenamiento, pero que generalice mejor que un único árbol.

    predicciones_train_forest = modelo_forest.predict(X)

    mae_train_forest = mean_absolute_error(
        y,
        predicciones_train_forest
    )


    # ----------------------------------------------------------
    # 6. EXTRAEMOS LA IMPORTANCIA DE LAS FEATURES
    # ----------------------------------------------------------
    #
    # Cada árbol calcula qué features fueron útiles para realizar
    # sus divisiones.
    #
    # Random Forest combina esa información entre todos los árboles
    # y nos ofrece una importancia global mediante:
    #
    #     feature_importances_
    #
    # Una importancia alta significa que esa variable ha contribuido
    # mucho a las divisiones realizadas por el conjunto de árboles.
    #
    # Una importancia baja significa que ha participado poco.
    #
    # Igual que antes:
    # importancia NO significa causalidad.

    importancia_forest = pd.Series(
        modelo_forest.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)


    # ----------------------------------------------------------
    # 7. CALCULAMOS INFORMACIÓN SOBRE LOS ÁRBOLES DEL BOSQUE
    # ----------------------------------------------------------
    #
    # estimators_ contiene todos los árboles entrenados
    # dentro del Random Forest.
    #
    # Podemos calcular la profundidad media de esos árboles
    # y el número medio de hojas.
    #
    # Esto será muy interesante para comparar con nuestro
    # Decision Tree individual:
    #
    #     profundidad = 21
    #     hojas = 225

    profundidades = [
        arbol.get_depth()
        for arbol in modelo_forest.estimators_
    ]

    hojas = [
        arbol.get_n_leaves()
        for arbol in modelo_forest.estimators_
    ]

    profundidad_media_forest = sum(profundidades) / len(profundidades)

    hojas_medias_forest = sum(hojas) / len(hojas)


    # ----------------------------------------------------------
    # 8. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # modelo_forest
    #     -> Random Forest entrenado con todos los datos.
    #
    # mae_forest
    #     -> MAE medio de Cross Validation.
    #
    # -scores_forest
    #     -> MAE de cada uno de los cinco folds.
    #
    # importancia_forest
    #     -> importancia asignada a cada feature.
    #
    # profundidad_media_forest
    #     -> profundidad media de los árboles del bosque.
    #
    # hojas_medias_forest
    #     -> número medio de hojas por árbol.
    #
    # mae_train_forest
    #     -> MAE sobre los datos utilizados para entrenar
    #        el modelo final.

    return (
        modelo_forest,
        mae_forest,
        -scores_forest,
        importancia_forest,
        profundidad_media_forest,
        hojas_medias_forest,
        mae_train_forest
    )

## 7.9. Extra Trees

In [14]:
# ==============================================================
# EXTRA TREES REGRESSOR
#
# Extra Trees construye muchos árboles de decisión y combina
# sus predicciones mediante un promedio.
#
# Se parece a Random Forest, pero introduce todavía más
# aleatoriedad en la construcción de los árboles.
#
# La idea es que, si los árboles son menos parecidos entre sí,
# sus errores estarán menos correlacionados.
#
# Al promediar muchos árboles diferentes podemos reducir
# la varianza del conjunto y mejorar la generalización.
# ==============================================================

def evalua_ExtraTrees(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # n_estimators indica cuántos árboles tendrá el conjunto.
    #
    # Usamos 200 árboles, igual que hicimos con Random Forest,
    # para que la comparación entre ambos modelos sea sencilla.
    #
    # No estamos optimizando todavía este parámetro.
    #
    # random_state=42 permite reproducir exactamente
    # los mismos resultados.
    #
    # n_jobs=-1 permite utilizar todos los núcleos disponibles
    # del procesador.
    #
    # No utilizamos StandardScaler:
    # los árboles no necesitan variables escaladas.
    #
    # A diferencia de Random Forest, Extra Trees introduce más
    # aleatoriedad en la elección de los puntos de corte.
    #
    # Eso hace que los árboles individuales puedan ser algo
    # menos "perfectos", pero también más diferentes entre sí.

    modelo_extratrees = ExtraTreesRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Utilizamos exactamente los mismos 5 folds que en los
    # modelos anteriores.
    #
    # Así mantenemos una comparación justa:
    #
    #     misma X
    #     mismo y
    #     mismos folds
    #     misma métrica MAE
    #
    # Lo único que cambia es el algoritmo.

    scores_extratrees = cross_val_score(
        modelo_extratrees,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO DE CROSS VALIDATION
    # ----------------------------------------------------------

    mae_extratrees = -scores_extratrees.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # Después de evaluar el modelo mediante Cross Validation,
    # lo entrenamos nuevamente utilizando todo X e y.
    #
    # Así tendremos un modelo final que podemos inspeccionar
    # y utilizar posteriormente para hacer predicciones.

    modelo_extratrees.fit(X, y)


    # ----------------------------------------------------------
    # 5. CALCULAMOS EL MAE SOBRE LOS DATOS DE ENTRENAMIENTO
    # ----------------------------------------------------------
    #
    # Igual que hicimos con Decision Tree y Random Forest.
    #
    # Comparamos:
    #
    #     MAE entrenamiento
    #             VS
    #     MAE Cross Validation
    #
    # Si el MAE de entrenamiento es muchísimo más bajo que
    # el MAE de CV, existe una señal de overfitting.
    #
    # Aquí queremos observar si Extra Trees ajusta todavía
    # más agresivamente que Random Forest y, sobre todo,
    # si ese ajuste se traduce o no en mejor generalización.

    predicciones_train_extratrees = modelo_extratrees.predict(X)

    mae_train_extratrees = mean_absolute_error(
        y,
        predicciones_train_extratrees
    )


    # ----------------------------------------------------------
    # 6. EXTRAEMOS LA IMPORTANCIA DE LAS FEATURES
    # ----------------------------------------------------------
    #
    # Extra Trees, igual que Random Forest, permite calcular
    # una importancia para cada feature mediante:
    #
    #     feature_importances_
    #
    # Una importancia alta significa que esa variable ha
    # contribuido mucho a las divisiones realizadas por
    # los árboles del conjunto.
    #
    # Una importancia baja significa que ha participado poco.
    #
    # IMPORTANTE:
    # importancia NO significa causalidad.

    importancia_extratrees = pd.Series(
        modelo_extratrees.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)


    # ----------------------------------------------------------
    # 7. CALCULAMOS INFORMACIÓN SOBRE LOS ÁRBOLES
    # ----------------------------------------------------------
    #
    # estimators_ contiene todos los árboles individuales
    # que forman Extra Trees.
    #
    # Calculamos:
    #
    #     profundidad media
    #     número medio de hojas
    #
    # Esto nos permitirá comparar directamente:
    #
    #     Decision Tree
    #     Random Forest
    #     Extra Trees

    profundidades_extratrees = [
        arbol.get_depth()
        for arbol in modelo_extratrees.estimators_
    ]

    hojas_extratrees = [
        arbol.get_n_leaves()
        for arbol in modelo_extratrees.estimators_
    ]

    profundidad_media_extratrees = (
        sum(profundidades_extratrees)
        / len(profundidades_extratrees)
    )

    hojas_medias_extratrees = (
        sum(hojas_extratrees)
        / len(hojas_extratrees)
    )


    # ----------------------------------------------------------
    # 8. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # modelo_extratrees
    #     -> Extra Trees entrenado con todos los datos.
    #
    # mae_extratrees
    #     -> MAE medio de Cross Validation.
    #
    # -scores_extratrees
    #     -> MAE individual de cada uno de los cinco folds.
    #
    # importancia_extratrees
    #     -> importancia asignada a cada feature.
    #
    # profundidad_media_extratrees
    #     -> profundidad media de los árboles.
    #
    # hojas_medias_extratrees
    #     -> número medio de hojas por árbol.
    #
    # mae_train_extratrees
    #     -> MAE sobre los datos de entrenamiento.

    return (
        modelo_extratrees,
        mae_extratrees,
        -scores_extratrees,
        importancia_extratrees,
        profundidad_media_extratrees,
        hojas_medias_extratrees,
        mae_train_extratrees
    )

## 7.10 Gradient Boosting

In [15]:
# ==============================================================
# GRADIENT BOOSTING REGRESSOR
#
# Gradient Boosting construye muchos árboles de forma secuencial.
#
# Cada nuevo árbol intenta corregir parte de los errores
# cometidos por el conjunto de árboles anteriores.
#
# A diferencia de Random Forest y Extra Trees:
#
#   - los árboles NO trabajan de forma independiente
#   - los árboles se construyen uno detrás de otro
#   - cada árbol aprende sobre los errores residuales anteriores
#
# La combinación progresiva de muchos árboles pequeños
# puede producir un modelo muy potente.
# ==============================================================

def evalua_GradientBoosting(X, y):

    # ----------------------------------------------------------
    # 1. CREAMOS EL MODELO
    # ----------------------------------------------------------
    #
    # n_estimators:
    #     número de árboles que se construirán secuencialmente.
    #
    # learning_rate:
    #     controla cuánto aporta cada nuevo árbol a la corrección.
    #
    #     Un learning_rate pequeño significa que cada árbol
    #     corrige poco, pero podemos acumular muchas correcciones.
    #
    # max_depth:
    #     controla la profundidad de cada árbol individual.
    #
    # En Gradient Boosting suele ser conveniente utilizar árboles
    # relativamente pequeños.
    #
    # Para este primer experimento usamos valores sencillos,
    # sin intentar optimizarlos.
    #
    # No utilizamos StandardScaler porque seguimos trabajando
    # con árboles.

    modelo_gradient = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    )


    # ----------------------------------------------------------
    # 2. VALIDACIÓN CRUZADA
    # ----------------------------------------------------------
    #
    # Utilizamos los mismos 5 folds y la misma métrica MAE
    # que con todos los modelos anteriores.
    #
    # Esto nos permite comparar directamente los resultados.

    scores_gradient = cross_val_score(
        modelo_gradient,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # ----------------------------------------------------------
    # 3. CALCULAMOS EL MAE MEDIO DE CROSS VALIDATION
    # ----------------------------------------------------------

    mae_gradient = -scores_gradient.mean()


    # ----------------------------------------------------------
    # 4. ENTRENAMOS EL MODELO CON TODOS LOS DATOS
    # ----------------------------------------------------------
    #
    # Después de evaluar mediante Cross Validation,
    # entrenamos el modelo utilizando todo X e y.
    #
    # Así podremos inspeccionar las importancias y utilizar
    # el modelo posteriormente para hacer recomendaciones.

    modelo_gradient.fit(X, y)


    # ----------------------------------------------------------
    # 5. CALCULAMOS EL MAE SOBRE ENTRENAMIENTO
    # ----------------------------------------------------------
    #
    # Igual que con nuestros otros modelos basados en árboles,
    # queremos comparar:
    #
    #     MAE entrenamiento
    #             VS
    #     MAE Cross Validation
    #
    # Esto nos permitirá observar si existe overfitting.

    predicciones_train_gradient = modelo_gradient.predict(X)

    mae_train_gradient = mean_absolute_error(
        y,
        predicciones_train_gradient
    )


    # ----------------------------------------------------------
    # 6. EXTRAEMOS LA IMPORTANCIA DE LAS FEATURES
    # ----------------------------------------------------------
    #
    # Gradient Boosting también proporciona:
    #
    #     feature_importances_
    #
    # Esta medida resume cuánto han contribuido las features
    # a las divisiones realizadas por todos los árboles.
    #
    # Igual que siempre:
    # importancia NO significa causalidad.

    importancia_gradient = pd.Series(
        modelo_gradient.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)


    profundidades_gradient = [arbol[0].get_depth() for arbol in modelo_gradient.estimators_]

    hojas_gradient = [arbol[0].get_n_leaves() for arbol in modelo_gradient.estimators_]

    # ----------------------------------------------------------
    # 7. DEVOLVEMOS LOS RESULTADOS
    # ----------------------------------------------------------
    #
    # modelo_gradient
    #     -> Gradient Boosting entrenado.
    #
    # mae_gradient
    #     -> MAE medio de Cross Validation.
    #
    # -scores_gradient
    #     -> MAE de cada fold.
    #
    # importancia_gradient
    #     -> importancia de cada feature.
    #
    # mae_train_gradient
    #     -> MAE sobre entrenamiento.

    return (modelo_gradient,
            mae_gradient,
        -scores_gradient,
        importancia_gradient,
        profundidades_gradient,
        hojas_gradient,
        mae_train_gradient
    )

## 7.11 Hist Gradient Boosting

In [17]:
def evalua_hist_gradient_boosting(X, y):

    # ============================================================
    # 1. CREACIÓN DEL MODELO
    # ============================================================

    # HistGradientBoostingRegressor es una versión moderna
    # de Gradient Boosting basada en histogramas.
    #
    # Igual que GradientBoostingRegressor:
    #
    # - crea árboles pequeños
    # - los crea de forma secuencial
    # - cada árbol intenta corregir los errores
    #   cometidos por los árboles anteriores
    #
    # La diferencia importante es que HistGradientBoosting
    # agrupa previamente los valores de las variables en "bins".
    #
    # De momento utilizamos los parámetros por defecto.
    # No queremos optimizar el modelo todavía:
    # primero queremos observar cómo se comporta.

    modelo_hist_gradient_boosting = HistGradientBoostingRegressor(random_state=42)


    # ============================================================
    # 2. VALIDACIÓN CRUZADA
    # ============================================================

    # Evaluamos el modelo mediante 5-fold Cross Validation.
    #
    # scoring='neg_mean_absolute_error'
    #
    # hace que sklearn devuelva el MAE en negativo.
    # Después simplemente cambiamos el signo.

    scores_hist_gradient_boosting = cross_val_score(
        modelo_hist_gradient_boosting,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # Convertimos los valores negativos en MAE positivos.

    folds_hist_gradient_boosting = (
        -scores_hist_gradient_boosting
    )


    # Calculamos el MAE medio de los cinco folds.

    mae_hist_gradient_boosting = (
        folds_hist_gradient_boosting.mean()
    )


    # ============================================================
    # 3. ENTRENAMIENTO FINAL
    # ============================================================

    # Una vez terminada la validación cruzada,
    # entrenamos el modelo con TODOS los juegos.
    #
    # Este es el modelo que utilizaremos posteriormente
    # para predecir los juegos no puntuados.

    modelo_hist_gradient_boosting.fit(
        X,
        y
    )


    # ============================================================
    # 4. ERROR SOBRE EL CONJUNTO DE ENTRENAMIENTO
    # ============================================================

    # Calculamos también el MAE sobre los mismos datos
    # con los que hemos entrenado.
    #
    # Esto NO mide la capacidad de generalización.
    #
    # Nos sirve para comparar:
    #
    # MAE entrenamiento
    #        VS
    # MAE Cross Validation
    #
    # y detectar posibles señales de sobreajuste.

    predicciones_train_hist_gradient_boosting = (
        modelo_hist_gradient_boosting.predict(X)
    )


    mae_train_hist_gradient_boosting = mean_absolute_error(
        y,
        predicciones_train_hist_gradient_boosting
    )


    # ============================================================
    # 5. NÚMERO DE ITERACIONES Y ÁRBOLES
    # ============================================================

    # n_iter_ indica cuántas iteraciones de boosting
    # ha realizado realmente el modelo.
    #
    # En un problema de regresión normalmente se construye
    # un árbol por iteración.

    numero_iteraciones_hist_gradient_boosting = (
        modelo_hist_gradient_boosting.n_iter_
    )


    # n_trees_per_iteration_ indica cuántos árboles
    # se construyen en cada iteración.
    #
    # Para regresión normalmente será 1.

    arboles_por_iteracion_hist_gradient_boosting = (
        modelo_hist_gradient_boosting.n_trees_per_iteration_
    )


    # Calculamos el número total de árboles.

    numero_arboles_hist_gradient_boosting = (
        numero_iteraciones_hist_gradient_boosting
        * arboles_por_iteracion_hist_gradient_boosting
    )


    # ============================================================
    # 6. PROFUNDIDAD Y NÚMERO DE HOJAS DE LOS ÁRBOLES
    # ============================================================

    # HistGradientBoostingRegressor no expone los árboles
    # mediante estimators_ como hace GradientBoostingRegressor.
    #
    # Para estudiar su estructura tenemos que acceder a
    # modelo._predictors.
    #
    # IMPORTANTE:
    #
    # _predictors es un atributo INTERNO de sklearn.
    # Lo utilizamos aquí únicamente con fines pedagógicos
    # para estudiar los árboles construidos por el modelo.
    #
    # Una futura versión de sklearn podría cambiar
    # esta estructura interna.

    profundidades = []
    numero_hojas = []


    # Cada elemento de _predictors corresponde
    # a una iteración del boosting.

    for iteracion in modelo_hist_gradient_boosting._predictors:

        # Dentro de cada iteración puede haber uno o más árboles.
        #
        # En nuestro problema de regresión normalmente habrá uno.

        for arbol in iteracion:

            # Los nodos del árbol están almacenados
            # en arbol.nodes.

            nodos = arbol.nodes


            # ----------------------------------------------------
            # PROFUNDIDAD DEL ÁRBOL
            # ----------------------------------------------------

            # Cada nodo almacena su profundidad.
            #
            # La profundidad del árbol será la máxima
            # profundidad encontrada entre todos sus nodos.

            profundidad_arbol = (
                nodos['depth'].max()
            )

            profundidades.append(
                profundidad_arbol
            )


            # ----------------------------------------------------
            # NÚMERO DE HOJAS
            # ----------------------------------------------------

            # is_leaf vale 1 cuando el nodo es una hoja.
            #
            # Sumando todos esos valores obtenemos
            # el número total de hojas del árbol.

            hojas_arbol = (
                nodos['is_leaf'].sum()
            )

            numero_hojas.append(
                hojas_arbol
            )


    # ============================================================
    # 7. ESTADÍSTICAS DE PROFUNDIDAD
    # ============================================================

    profundidad_media_hist_gradient_boosting = (
        np.mean(profundidades)
    )

    profundidad_minima_hist_gradient_boosting = (
        np.min(profundidades)
    )

    profundidad_maxima_hist_gradient_boosting = (
        np.max(profundidades)
    )


    # ============================================================
    # 8. ESTADÍSTICAS DEL NÚMERO DE HOJAS
    # ============================================================

    hojas_medias_hist_gradient_boosting = (
        np.mean(numero_hojas)
    )

    hojas_minimas_hist_gradient_boosting = (
        np.min(numero_hojas)
    )

    hojas_maximas_hist_gradient_boosting = (
        np.max(numero_hojas)
    )


    # ============================================================
    # 9. DEVOLVEMOS TODOS LOS RESULTADOS
    # ============================================================

    return (
        modelo_hist_gradient_boosting,
        mae_hist_gradient_boosting,
        folds_hist_gradient_boosting,
        mae_train_hist_gradient_boosting,
        numero_iteraciones_hist_gradient_boosting,
        numero_arboles_hist_gradient_boosting,
        profundidad_media_hist_gradient_boosting,
        profundidad_minima_hist_gradient_boosting,
        profundidad_maxima_hist_gradient_boosting,
        hojas_medias_hist_gradient_boosting,
        hojas_minimas_hist_gradient_boosting,
        hojas_maximas_hist_gradient_boosting
    )

## 7.11 XGBoost

In [35]:


def evalua_xgboost(X, y):

    # ============================================================
    # 1. CREACIÓN DEL MODELO
    # ============================================================

    # XGBoost significa:
    #
    # Extreme Gradient Boosting
    #
    # Igual que GradientBoostingRegressor, construye árboles
    # de forma SECUENCIAL:
    #
    # árbol 1
    #     ↓
    # árbol 2 corrige errores del árbol 1
    #     ↓
    # árbol 3 corrige errores de los anteriores
    #     ↓
    # ...
    #
    # Sin embargo, XGBoost incorpora varias mejoras:
    #
    # - regularización
    # - utilización del gradiente y del hessiano
    # - control de la complejidad de los árboles
    # - mecanismos para reducir el sobreajuste
    # - una implementación especialmente optimizada
    #
    # De momento NO vamos a tunear hiperparámetros.
    #
    # Queremos observar primero cómo se comporta
    # XGBoost prácticamente con su configuración estándar.

    # Primero formateamos X para xgboost con la funcion anterior
    X_xgboost = prepara_X_para_modelo(X,'XGBoost')

    modelo_xgboost = XGBRegressor(
        random_state=42,
        n_jobs=-1
    )


    # ============================================================
    # 2. VALIDACIÓN CRUZADA
    # ============================================================

    # Evaluamos exactamente igual que los demás modelos:
    #
    # 5-fold Cross Validation.
    #
    # En cada fold:
    #
    # - se entrena con aproximadamente el 80 % de los juegos
    # - se prueba con aproximadamente el 20 % restante
    #
    # Esto nos permite estimar mejor la capacidad
    # del modelo para generalizar a juegos que no ha visto.

    scores_xgboost = cross_val_score(
        modelo_xgboost,
        X_xgboost,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # sklearn devuelve el MAE en negativo.
    #
    # Cambiamos el signo para obtener
    # los valores normales del MAE.
    folds_xgboost = -scores_xgboost


    # Calculamos el MAE medio de los cinco folds.
    mae_xgboost = folds_xgboost.mean()


    # ============================================================
    # 3. ENTRENAMIENTO FINAL
    # ============================================================

    # Una vez terminada la validación cruzada,
    # entrenamos XGBoost utilizando TODOS los juegos.
    #
    # Este será posteriormente el modelo que utilizaremos
    # para predecir los juegos que todavía no hemos puntuado.
    modelo_xgboost.fit(
        X_xgboost,
        y
    )


    # ============================================================
    # 4. MAE SOBRE LOS DATOS DE ENTRENAMIENTO
    # ============================================================

    # Ahora hacemos predicciones sobre los mismos juegos
    # utilizados para entrenar el modelo.
    #
    # IMPORTANTE:
    #
    # Este MAE NO sirve para medir la capacidad
    # real de generalización.
    #
    # Para eso tenemos el MAE de Cross Validation.
    #
    # Sin embargo, nos resulta muy útil para detectar
    # posibles señales de sobreajuste.
    #
    # Por ejemplo:
    #
    # MAE entrenamiento muy pequeño
    # MAE Cross Validation mucho mayor
    #
    # sería una señal de que el modelo está
    # aprendiendo demasiado específicamente
    # los juegos utilizados durante el entrenamiento.

    predicciones_train_xgboost = modelo_xgboost.predict(X_xgboost)


    mae_train_xgboost = mean_absolute_error(
        y,
        predicciones_train_xgboost
    )


    # ============================================================
    # 5. EXTRAEMOS LOS ÁRBOLES CONSTRUIDOS POR XGBOOST
    # ============================================================

    # modelo_xgboost es el modelo de sklearn.
    #
    # Pero XGBoost mantiene internamente un objeto llamado
    # "Booster", que contiene los árboles realmente construidos.
    #
    # get_booster() nos permite acceder a ellos.

    booster_xgboost = modelo_xgboost.get_booster()


    # get_dump() devuelve una representación textual
    # de cada árbol.
    #
    # Cada elemento de esta lista corresponde a un árbol.

    arboles_xgboost = booster_xgboost.get_dump()


    # Número total de árboles construidos.

    numero_arboles_xgboost = len(arboles_xgboost)


    # ============================================================
    # 6. PROFUNDIDAD Y NÚMERO DE HOJAS
    # ============================================================

    # Vamos a estudiar la estructura de cada árbol.
    #
    # XGBoost representa los árboles internamente
    # mediante líneas de texto.
    #
    # La indentación mediante tabuladores nos indica
    # a qué profundidad se encuentra cada nodo.
    #
    # Además, una hoja contiene el texto:
    #
    # ":leaf="
    #
    # Gracias a esto podemos reconstruir:
    #
    # - profundidad de cada árbol
    # - número de hojas de cada árbol

    profundidades_xgboost = []
    numero_hojas_xgboost = []


    # Recorremos todos los árboles.

    for arbol in arboles_xgboost:

        # Dividimos la representación del árbol
        # en líneas individuales.

        lineas = arbol.splitlines()

        # Guardaremos las profundidades
        # de todos los nodos de este árbol.
        profundidades_nodos = []


        # Contador de hojas de este árbol.
        hojas_arbol = 0


        # Recorremos todos los nodos del árbol.
        for linea in lineas:

            # XGBoost utiliza tabuladores para indicar
            # la profundidad del nodo.
            #
            # Nodo raíz:
            #
            # 0 tabuladores
            #
            # Siguiente nivel:
            #
            # 1 tabulador
            #
            # etc.

            profundidad_nodo = (
                len(linea) - len(linea.lstrip('\t'))
            )


            profundidades_nodos.append(
                profundidad_nodo
            )


            # Si encontramos ":leaf=" significa
            # que este nodo es una hoja.

            if ':leaf=' in linea:
                hojas_arbol += 1

        # La profundidad total del árbol será
        # la profundidad máxima encontrada.

        profundidad_arbol = max(
            profundidades_nodos
        )

        profundidades_xgboost.append(
            profundidad_arbol
        )

        numero_hojas_xgboost.append(
            hojas_arbol
        )


    # ============================================================
    # 7. ESTADÍSTICAS DE PROFUNDIDAD
    # ============================================================

    # Calculamos:
    #
    # - profundidad media
    # - profundidad mínima
    # - profundidad máxima
    #
    # para poder compararlo directamente con
    # Gradient Boosting e Hist Gradient Boosting.

    profundidad_media_xgboost = np.mean(
        profundidades_xgboost
    )


    profundidad_minima_xgboost = np.min(
        profundidades_xgboost
    )


    profundidad_maxima_xgboost = np.max(
        profundidades_xgboost
    )


    # ============================================================
    # 8. ESTADÍSTICAS DEL NÚMERO DE HOJAS
    # ============================================================

    hojas_medias_xgboost = np.mean(
        numero_hojas_xgboost
    )


    hojas_minimas_xgboost = np.min(
        numero_hojas_xgboost
    )


    hojas_maximas_xgboost = np.max(
        numero_hojas_xgboost
    )


    # ============================================================
    # 9. DEVOLVEMOS TODOS LOS RESULTADOS
    # ============================================================

    return (
        modelo_xgboost,
        mae_xgboost,
        folds_xgboost,
        mae_train_xgboost,
        numero_arboles_xgboost,
        profundidad_media_xgboost,
        profundidad_minima_xgboost,
        profundidad_maxima_xgboost,
        hojas_medias_xgboost,
        hojas_minimas_xgboost,
        hojas_maximas_xgboost
    )

## 7.12 LightGBM

In [36]:
def evalua_lightgbm(X, y):

    # ============================================================
    # 1. PREPARAMOS LAS FEATURES PARA LIGHTGBM
    # ============================================================

    # Creamos una versión específica de X para LightGBM.
    #
    # El dataframe X original NO se modifica.

    X_lightgbm = prepara_X_para_modelo(X,'LightGBM')


    # ============================================================
    # 2. CREACIÓN DEL MODELO
    # ============================================================

    # LightGBM significa:
    #
    # Light Gradient Boosting Machine
    #
    # Igual que Gradient Boosting y XGBoost,
    # construye árboles de forma SECUENCIAL.
    #
    # Cada nuevo árbol intenta corregir los errores
    # que todavía comete el conjunto de árboles anteriores.
    #
    # Una característica importante de LightGBM es que
    # normalmente hace crecer los árboles "leaf-wise":
    #
    # busca la hoja cuya división puede reducir más el error
    # y continúa desarrollando esa hoja.
    #
    # Esto puede producir árboles muy eficientes y potentes,
    # pero también puede favorecer el sobreajuste
    # cuando tenemos muy pocos ejemplos.
    #
    # De momento NO hacemos tuning.
    #
    # Queremos observar su comportamiento prácticamente
    # con los parámetros estándar.

    modelo_lightgbm = LGBMRegressor(
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )


    # ============================================================
    # 3. VALIDACIÓN CRUZADA
    # ============================================================

    # Utilizamos exactamente la misma metodología
    # que para todos los modelos anteriores:
    #
    # 5-fold Cross Validation.

    scores_lightgbm = cross_val_score(
        modelo_lightgbm,
        X_lightgbm,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # sklearn devuelve el MAE en negativo.
    # Cambiamos el signo.

    folds_lightgbm = -scores_lightgbm

    # MAE medio de los cinco folds.

    mae_lightgbm = folds_lightgbm.mean()


    # ============================================================
    # 4. ENTRENAMIENTO FINAL
    # ============================================================

    # Entrenamos ahora el modelo con TODOS los juegos.
    #
    # Este será el modelo utilizado posteriormente
    # para realizar recomendaciones.

    modelo_lightgbm.fit(
        X_lightgbm,
        y
    )


    # ============================================================
    # 5. MAE SOBRE ENTRENAMIENTO
    # ============================================================

    # Calculamos también el MAE sobre los propios
    # datos de entrenamiento.
    #
    # Compararemos:
    #
    # MAE entrenamiento
    # VS
    # MAE Cross Validation
    #
    # para detectar posible sobreajuste.

    predicciones_train_lightgbm = (
        modelo_lightgbm.predict(
            X_lightgbm
        )
    )


    mae_train_lightgbm = mean_absolute_error(
        y,
        predicciones_train_lightgbm
    )


    # ============================================================
    # 6. ACCESO A LOS ÁRBOLES INTERNOS DE LIGHTGBM
    # ============================================================

    # LightGBM guarda los árboles entrenados dentro
    # de su objeto interno Booster.
    #
    # dump_model() devuelve la estructura completa
    # del modelo en forma de diccionario.

    estructura_modelo = (
        modelo_lightgbm
        .booster_
        .dump_model()
    )


    # tree_info contiene todos los árboles.

    arboles_lightgbm = estructura_modelo['tree_info']


    # Número total de árboles.

    numero_arboles_lightgbm = len(
        arboles_lightgbm
    )


    # ============================================================
    # 7. FUNCIÓN AUXILIAR PARA ANALIZAR CADA ÁRBOL
    # ============================================================

    # Los árboles de LightGBM están almacenados
    # como estructuras anidadas.
    #
    # Utilizamos una pequeña función recursiva
    # para calcular:
    #
    # - profundidad
    # - número de hojas

    def analiza_arbol(nodo):

        # Si encontramos "leaf_value",
        # hemos llegado a una hoja.

        if 'leaf_value' in nodo:

            return 0, 1


        # Analizamos la rama izquierda.

        profundidad_izquierda, hojas_izquierda = (
            analiza_arbol(
                nodo['left_child']
            )
        )


        # Analizamos la rama derecha.

        profundidad_derecha, hojas_derecha = (
            analiza_arbol(
                nodo['right_child']
            )
        )


        # La profundidad será:
        #
        # 1 + la mayor profundidad de las dos ramas.

        profundidad = (
            1
            + max(
                profundidad_izquierda,
                profundidad_derecha
            )
        )


        # El número de hojas será la suma
        # de las hojas de ambas ramas.

        hojas = (
            hojas_izquierda
            + hojas_derecha
        )


        return profundidad, hojas


    # ============================================================
    # 8. ANALIZAMOS TODOS LOS ÁRBOLES
    # ============================================================

    profundidades_lightgbm = []
    numero_hojas_lightgbm = []


    for arbol in arboles_lightgbm:

        profundidad, hojas = analiza_arbol(
            arbol['tree_structure']
        )


        profundidades_lightgbm.append(
            profundidad
        )


        numero_hojas_lightgbm.append(
            hojas
        )


    # ============================================================
    # 9. ESTADÍSTICAS DE PROFUNDIDAD
    # ============================================================

    profundidad_media_lightgbm = np.mean(
        profundidades_lightgbm
    )

    profundidad_minima_lightgbm = np.min(
        profundidades_lightgbm
    )

    profundidad_maxima_lightgbm = np.max(
        profundidades_lightgbm
    )


    # ============================================================
    # 10. ESTADÍSTICAS DEL NÚMERO DE HOJAS
    # ============================================================

    hojas_medias_lightgbm = np.mean(
        numero_hojas_lightgbm
    )

    hojas_minimas_lightgbm = np.min(
        numero_hojas_lightgbm
    )

    hojas_maximas_lightgbm = np.max(
        numero_hojas_lightgbm
    )


    # ============================================================
    # 11. DEVOLVEMOS LOS RESULTADOS
    # ============================================================

    return (
        modelo_lightgbm,
        mae_lightgbm,
        folds_lightgbm,
        mae_train_lightgbm,
        numero_arboles_lightgbm,
        profundidad_media_lightgbm,
        profundidad_minima_lightgbm,
        profundidad_maxima_lightgbm,
        hojas_medias_lightgbm,
        hojas_minimas_lightgbm,
        hojas_maximas_lightgbm
    )

## 7.13 CatBoost

In [46]:
def evalua_catboost(X, y):

    # ============================================================
    # 1. PREPARAMOS X
    # ============================================================

    # Utilizamos nuestra función general de preparación.
    #
    # En principio CatBoost no necesita ningún cambio especial
    # en nuestro dataset actual, pero mantenemos la misma
    # arquitectura que con XGBoost y LightGBM.

    X_catboost = prepara_X_para_modelo(X,'CatBoost')

    # ============================================================
    # 2. CREACIÓN DEL MODELO
    # ============================================================

    # CatBoost significa:
    #
    # Categorical Boosting
    #
    # Es otro algoritmo de Gradient Boosting basado en árboles.
    #
    # Se diseñó especialmente para:
    #
    # - trabajar muy bien con variables categóricas
    # - reducir determinados tipos de sobreajuste
    # - evitar leakage en transformaciones categóricas
    # - construir modelos robustos con poca preparación manual
    #
    # En nuestro caso las categorías ya están codificadas,
    # así que no aprovechamos todavía todo su potencial
    # categórico.
    #
    # De momento NO hacemos tuning.
    #
    # Queremos ver cómo se comporta con una configuración
    # prácticamente estándar.

    modelo_catboost = CatBoostRegressor(random_seed=42,verbose=0)


    # ============================================================
    # 3. VALIDACIÓN CRUZADA
    # ============================================================

    # Evaluamos con 5-fold Cross Validation,
    # igual que todos los modelos anteriores.

    scores_catboost = cross_val_score(
        modelo_catboost,
        X_catboost,
        y,
        cv=5,
        scoring='neg_mean_absolute_error'
    )


    # sklearn devuelve MAE negativo.

    folds_catboost = -scores_catboost


    # MAE medio.

    mae_catboost = folds_catboost.mean()


    # ============================================================
    # 4. ENTRENAMIENTO FINAL
    # ============================================================

    # Entrenamos el modelo con todos los datos
    # una vez terminada la validación cruzada.

    modelo_catboost.fit(
        X_catboost,
        y
    )


    # ============================================================
    # 5. MAE DE ENTRENAMIENTO
    # ============================================================

    predicciones_train_catboost = (
        modelo_catboost.predict(
            X_catboost
        )
    )


    mae_train_catboost = mean_absolute_error(
        y,
        predicciones_train_catboost
    )


    # ============================================================
    # 6. NÚMERO DE ÁRBOLES
    # ============================================================

    # CatBoost llama "tree_count_" al número de árboles
    # que realmente han sido construidos.

    numero_arboles_catboost = (
        modelo_catboost.tree_count_
    )


    # ============================================================
    # 7. PROFUNDIDAD DE LOS ÁRBOLES
    # ============================================================

    # Con la configuración estándar, CatBoost utiliza
    # árboles simétricos.
    #
    # La profundidad configurada puede obtenerse de
    # los parámetros del modelo.

    parametros_catboost = (
        modelo_catboost.get_params()
    )


    profundidad_catboost = parametros_catboost.get(
        'depth',
        6
    )


    # ============================================================
    # 8. NÚMERO TEÓRICO DE HOJAS
    # ============================================================

    # Los árboles simétricos de CatBoost tienen,
    # en condiciones normales:
    #
    # 2 ^ profundidad
    #
    # hojas.
    #
    # Por ejemplo:
    #
    # profundidad 6
    # → 64 hojas teóricas.

    hojas_teoricas_catboost = (
        2 ** profundidad_catboost
    )


    # ============================================================
    # 9. DEVOLVEMOS RESULTADOS
    # ============================================================

    return (
        modelo_catboost,
        mae_catboost,
        folds_catboost,
        mae_train_catboost,
        numero_arboles_catboost,
        profundidad_catboost,
        hojas_teoricas_catboost
    )

# **PASO 8.CREAMOS LAS LISTAS DE FEATURES**

In [33]:
# LANZAMOS EL MACHINE LEARNING
# Creamos la matriz de parámetros X y el vector y de respuestas.

features_base = [
    'GameWeight', 'MinPlayers', 'MaxPlayers', 'MfgPlaytime',
    'ComMinPlaytime', 'ComMaxPlaytime', 'ComAgeRec', 'LanguageEase',
    'Cat:Thematic', 'Cat:Strategy', 'Cat:War', 'Cat:Family',
    'Cat:CGS', 'Cat:Abstract', 'Cat:Party', 'Cat:Childrens'
]

features_games = [
   # 'YearPublished', FEATURE PROBLEMATICA
    'GameWeight',
    'MinPlayers',
    'MaxPlayers',
    'MfgPlaytime',
    'ComMinPlaytime',
    'ComMaxPlaytime',
    'ComAgeRec',
    'LanguageEase',
    'MfgAgeRec',
    'NumAlternates',
    'NumExpansions',
    'NumImplementations',
    'IsReimplementation',
    'Kickstarted',
    'Cat:Thematic',
    'Cat:Strategy',
    'Cat:War',
    'Cat:Family',
    'Cat:CGS',
    'Cat:Abstract',
    'Cat:Party',
    'Cat:Childrens'
]

# Creamos la lista de features de base + mechanics
features_mechanics = [
    columna
    for columna in df_mechanics.columns
    if columna != 'BGGId'
]
features_games_mechanics = features_games + features_mechanics

# Añadimos los features_games_themes
features_themes = [
    columna
    for columna in df_themes.columns
    if columna != 'BGGId'
]
features_games_themes = features_games_mechanics + features_themes

# Añadimos los features_games_subcategories
features_subcategories = [
    columna
    for columna in df_subcategories.columns
    if columna != 'BGGId'
]
features_games_subcategories = features_games_themes + features_subcategories

# Añadimos los features_games_designers
features_designers = [
    columna
    for columna in df_designers.columns
    if columna != 'BGGId'
]
features_games_designers = features_games_subcategories + features_designers

# Añadimos los publishers
features_publishers = [
    columna
    for columna in df_publishers.columns
    if columna != 'BGGId'
]
features_games_publishers = features_games_designers + features_publishers


print("Número de features de base:", len(features_games))
print("Número de features con mecánicas:", len(features_games_mechanics))
print("Número de features de themes:", len(features_games_themes))
print("Número de features con subcategorias:", len(features_games_subcategories))
print("Número de features con designers:", len(features_games_designers))
print("Número de features con publishers:", len(features_games_publishers))

# features_sin_variacion = [
#     columna
#     for columna in features_games_mechanics
#     if df_train_mechanics[columna].nunique() <= 1
# ]

# print("Features sin variación:", len(features_sin_variacion))

# for feature in features_sin_variacion:
#     print(feature)


Número de features de base: 22
Número de features con mecánicas: 179
Número de features de themes: 396
Número de features con subcategorias: 406
Número de features con designers: 1999
Número de features con publishers: 3864


# **PASO 8. INVOCAMOS LOS ENTRENAMIENTOS CON CADA UNO DE LOS MODELOS. EL RESULTADO ES EL CONJUNTO DE MODELOS ENTRENADOS**

In [47]:
# INVOCAMOS LOS MODELOS PARA ENTRENAR
#Generamos la matriz X de features y el vector "y" de mis evaluaciones

#X = df_train[features_games]
#y = df_train['UserRating']

#X = df_train_mechanics[features_games_mechanics]
#y = df_train_mechanics['UserRating']

#X = df_train_themes[features_games_themes]
#y = df_train_themes['UserRating']

#X = df_train_subcategories[features_games_subcategories]
#y = df_train_subcategories['UserRating']

#X = df_train_designers[features_games_designers]
#y = df_train_designers['UserRating']

X = df_train_publishers[features_games_publishers]
y = df_train_publishers['UserRating']

# ==============================================================
# EVALUAMOS LOS MODELOS
# ==============================================================

# Dummy devuelve:
#   MAE medio
#   MAE de cada fold
mae_dummy, folds_dummy = evalua_dummy(X, y)
print("Dummy entrenado")

# Linear Regression devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   coeficientes
#   intercepto
modelo_linear, mae_linear, folds_linear, pesos_linear, intercepto_linear = \
    evalua_LinearRegression(X, y)
print("Linear Regressor entrenado")

# Ridge devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   coeficientes
#   intercepto
modelo_ridge, mae_ridge, folds_ridge, pesos_ridge, intercepto_ridge = \
    evalua_Ridge(X, y)
print ("Ridge entrenado")

# Huber regressor devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   coeficientes
#   intercepto
#modelo_huber, mae_huber, folds_huber, pesos_huber, intercepto_huber = evalua_Huber(X, y)
#print ("Huber entrenado")

# Lasso devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   coeficientes
#   intercepto
#   features que Lasso ha decidido eliminar poniendo coeficiente "w" a 0
modelo_lasso, mae_lasso, folds_lasso, pesos_lasso, intercepto_lasso, features_eliminadas_lasso = evalua_Lasso(X, y)
print ("Lasso entrenado")

# Elastic Net devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   coeficientes
#   intercepto
#   features que Lasso ha decidido eliminar poniendo coeficiente "w" a 0
modelo_elasticnet, mae_elasticnet, folds_elasticnet, pesos_elasticnet, intercepto_elasticnet, features_eliminadas_elasticnet = evalua_ElasticNet(X, y)
print ("Elastic Net entrenado")

# DecissionTree devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   importancia asignada a cada feature
#   profundidad del arbol
#   hojas del árbol
modelo_tree, mae_tree, folds_tree, importancia_tree, profundidad_tree, hojas_tree, mae_train_tree = evalua_DecisionTree(X, y)
print ("Decision Tree entrenado")

# Random Forest devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   importancia asignada a cada feature
#   profundidad del arbol
#   hojas del árbol
modelo_forest, mae_forest, folds_forest, importancia_forest, profundidad_media_forest, hojas_medias_forest, mae_train_forest = evalua_RandomForest(X, y)
print ("Random Forest entrenado")

# ExtraTrees devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   importancia asignada a cada feature
#   profundidad del arbol
#   hojas del árbol
#   MAE del entrenamiento
modelo_extratrees, mae_extratrees, folds_extratrees, importancia_extratrees, profundidad_media_extratrees, hojas_medias_extratrees, mae_train_extratrees = evalua_ExtraTrees(X, y)
print ("Extra Trees entrenado")

# Gradient Boosting devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   importancia asignada a cada feature
#   profundidad del arbol
#   hojas del árbol#   MAE del entrenamiento

modelo_gradient, mae_gradient, folds_gradient, importancia_gradient, profundidades_gradient, hojas_gradient, mae_train_gradient = evalua_GradientBoosting(X, y)
print ("Gradient Boosting entrenado")

# Hist Gradient Boosting devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   MAE del entrenamiento
#   Numero de iteraciones
#   Numero de arboles
#   profundidad media
#   profundidad minima
#   profundidad maxima
#   hojas medias
#   hojas minimas
#   hojas maximas
(
    modelo_hist_gradient_boosting,
    mae_hist_gradient_boosting,
    folds_hist_gradient_boosting,
    mae_train_hist_gradient_boosting,
    numero_iteraciones_hist_gradient_boosting,
    numero_arboles_hist_gradient_boosting,
    profundidad_media_hist_gradient_boosting,
    profundidad_minima_hist_gradient_boosting,
    profundidad_maxima_hist_gradient_boosting,
    hojas_medias_hist_gradient_boosting,
    hojas_minimas_hist_gradient_boosting,
    hojas_maximas_hist_gradient_boosting
) = evalua_hist_gradient_boosting(X, y)
print ("Hist Gradient Boosting entrenado")

# Hist XGBoost devuelve:
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   MAE del entrenamiento
#   Numero de arboles
#   profundidad media
#   profundidad minima
#   profundidad maxima
#   hojas medias
#   hojas minimas
#   hojas maximas
(
    modelo_xgboost,
    mae_xgboost,
    folds_xgboost,
    mae_train_xgboost,
    numero_arboles_xgboost,
    profundidad_media_xgboost,
    profundidad_minima_xgboost,
    profundidad_maxima_xgboost,
    hojas_medias_xgboost,
    hojas_minimas_xgboost,
    hojas_maximas_xgboost
) = evalua_xgboost(X,y)
print ("XGBoost entrenado")

# LightGBM devuelve
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   MAE del entrenamiento
#   Numero de arboles
#   profundidad media
#   profundidad minima
#   profundidad maxima
#   hojas medias
#   hojas minimas
#   hojas maximas
(
    modelo_lightgbm,
    mae_lightgbm,
    folds_lightgbm,
    mae_train_lightgbm,
    numero_arboles_lightgbm,
    profundidad_media_lightgbm,
    profundidad_minima_lightgbm,
    profundidad_maxima_lightgbm,
    hojas_medias_lightgbm,
    hojas_minimas_lightgbm,
    hojas_maximas_lightgbm
) = evalua_lightgbm(X,y)
print ("LightGBM entrenado")

# CatBoost devuelve
#   modelo entrenado
#   MAE medio
#   MAE de cada fold
#   MAE del entrenamiento
#   Numero de arboles
#   profundidad
#   hojas teoricas
(
    modelo_catboost,
    mae_catboost,
    folds_catboost,
    mae_train_catboost,
    numero_arboles_catboost,
    profundidad_catboost,
    hojas_teoricas_catboost
) = evalua_catboost(X,y)
print("CatBoost entrenado")

print ("ENTRENAMIENTO DE MODELOS TERMINADO")

Dummy entrenado
Linear Regressor entrenado
Ridge entrenado
Lasso entrenado
Elastic Net entrenado
Decision Tree entrenado
Random Forest entrenado
Extra Trees entrenado
Gradient Boosting entrenado
Hist Gradient Boosting entrenado
XGBoost entrenado
LightGBM entrenado
CatBoost entrenado
ENTRENAMIENTO DE MODELOS TERMINADO


# **PASO 9: PRESENTACION EN PANTALLA DE LOS RESULTADOS DEL APRENDIZAJE**

In [49]:
# ==============================================================
# PRESENTACIÓN DE RESULTADOS DE APRENDIZAJE
# ==============================================================

# Calculamos cuánto mejora cada modelo frente al baseline Dummy.
# Una mejora positiva significa que el modelo tiene menor MAE
# que simplemente predecir siempre la nota media.

mejora_linear = mae_dummy - mae_linear
mejora_ridge = mae_dummy - mae_ridge
#mejora_huber = mae_dummy - mae_huber
mejora_lasso = mae_dummy - mae_lasso
mejora_elasticnet = mae_dummy - mae_elasticnet
mejora_tree = mae_dummy - mae_tree
mejora_forest = mae_dummy - mae_forest
mejora_extratrees = mae_dummy - mae_extratrees
mejora_gradient = mae_dummy - mae_gradient
mejora_hist_gradient_boosting = mae_dummy - mae_hist_gradient_boosting
mejora_xgboost = mae_dummy - mae_xgboost
mejora_lightgbm = mae_dummy - mae_lightgbm
mejora_catboost = mae_dummy - mae_catboost

# ==============================================================
# RESULTADOS DE LA COMPARACIÓN DE LOS MODELOS
# ==============================================================


print()
print("=" * 78)
print("COMPARACIÓN DEL RENDIMIENTO DE LOS MODELOS")
print("=" * 78)

print(
    f"{'MODELO':<22}"
    f"{'MAE':>10}"
    f"{'MEJORA VS DUMMY':>20}"
    f"{'INTERCEPTO':>16}"
)

print("-" * 78)

print(
    f"{'Dummy (media)':<22}"
    f"{mae_dummy:>10.3f}"
    f"{0.0:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'Linear Regression':<22}"
    f"{mae_linear:>10.3f}"
    f"{mejora_linear:>+20.3f}"
    f"{intercepto_linear:>16.3f}"
)

print(
    f"{'Ridge':<22}"
    f"{mae_ridge:>10.3f}"
    f"{mejora_ridge:>+20.3f}"
    f"{intercepto_ridge:>16.3f}"
)

# print(
#     f"{'Huber':<22}"
#     f"{mae_huber:>10.3f}"
#     f"{mejora_huber:>+20.3f}"
#     f"{intercepto_huber:>16.3f}"
# )

print(
    f"{'Lasso':<22}"
    f"{mae_lasso:>10.3f}"
    f"{mejora_lasso:>+20.3f}"
    f"{intercepto_lasso:>16.3f}"
)

print(
    f"{'Elastic Net':<22}"
    f"{mae_elasticnet:>10.3f}"
    f"{mejora_elasticnet:>+20.3f}"
    f"{intercepto_elasticnet:>16.3f}"
)

print(
    f"{'Decission Tree':<22}"
    f"{mae_tree:>10.3f}"
    f"{mejora_tree:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'Random Forest':<22}"
    f"{mae_forest:>10.3f}"
    f"{mejora_forest:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'Extra Trees':<22}"
    f"{mae_extratrees:>10.3f}"
    f"{mejora_extratrees:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'Gradient Boosting':<22}"
    f"{mae_gradient:>10.3f}"
    f"{mejora_gradient:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'Hist Gradient Boosting':<22}"
    f"{mae_hist_gradient_boosting:>10.3f}"
    f"{mejora_hist_gradient_boosting:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'XGBoost':<22}"
    f"{mae_xgboost:>10.3f}"
    f"{mejora_xgboost:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'LightGBM':<22}"
    f"{mae_lightgbm:>10.3f}"
    f"{mejora_lightgbm:>+20.3f}"
    f"{'-':>16}"
)

print(
    f"{'CatBoost':<22}"
    f"{mae_catboost:>10.3f}"
    f"{mejora_catboost:>+20.3f}"
    f"{'-':>16}"
)

print("=" * 78)
print()
print("Cuanto menor es el MAE, mejor predice el modelo.")
print("Una mejora positiva significa que el modelo supera al Dummy.")


# ==============================================================
# RESULTADOS DE CADA FOLD
# ==============================================================

print()
print("=" * 86)
print("MAE EN CADA FOLD DE LA VALIDACIÓN CRUZADA")
print("=" * 86)

print(
    f"{'MODELO':<22}"
    f"{'FOLD 1':>10}"
    f"{'FOLD 2':>10}"
    f"{'FOLD 3':>10}"
    f"{'FOLD 4':>10}"
    f"{'FOLD 5':>10}"
)

print("-" * 86)

print(
    f"{'Dummy':<22}"
    f"{folds_dummy[0]:>10.3f}"
    f"{folds_dummy[1]:>10.3f}"
    f"{folds_dummy[2]:>10.3f}"
    f"{folds_dummy[3]:>10.3f}"
    f"{folds_dummy[4]:>10.3f}"
)

print(
    f"{'Linear Regression':<22}"
    f"{folds_linear[0]:>10.3f}"
    f"{folds_linear[1]:>10.3f}"
    f"{folds_linear[2]:>10.3f}"
    f"{folds_linear[3]:>10.3f}"
    f"{folds_linear[4]:>10.3f}"
)

print(
    f"{'Ridge':<22}"
    f"{folds_ridge[0]:>10.3f}"
    f"{folds_ridge[1]:>10.3f}"
    f"{folds_ridge[2]:>10.3f}"
    f"{folds_ridge[3]:>10.3f}"
    f"{folds_ridge[4]:>10.3f}"
)

# print(
#     f"{'Huber':<22}"
#     f"{folds_huber[0]:>10.3f}"
#     f"{folds_huber[1]:>10.3f}"
#     f"{folds_huber[2]:>10.3f}"
#     f"{folds_huber[3]:>10.3f}"
#     f"{folds_huber[4]:>10.3f}"
# )

print(
    f"{'Lasso':<22}"
    f"{folds_lasso[0]:>10.3f}"
    f"{folds_lasso[1]:>10.3f}"
    f"{folds_lasso[2]:>10.3f}"
    f"{folds_lasso[3]:>10.3f}"
    f"{folds_lasso[4]:>10.3f}"
)

print(
    f"{'Elastic Net':<22}"
    f"{folds_elasticnet[0]:>10.3f}"
    f"{folds_elasticnet[1]:>10.3f}"
    f"{folds_elasticnet[2]:>10.3f}"
    f"{folds_elasticnet[3]:>10.3f}"
    f"{folds_elasticnet[4]:>10.3f}"
)

print(
    f"{'Decission Tree':<22}"
    f"{folds_tree[0]:>10.3f}"
    f"{folds_tree[1]:>10.3f}"
    f"{folds_tree[2]:>10.3f}"
    f"{folds_tree[3]:>10.3f}"
    f"{folds_tree[4]:>10.3f}"
)

print(
    f"{'Random Forest':<22}"
    f"{folds_forest[0]:>10.3f}"
    f"{folds_forest[1]:>10.3f}"
    f"{folds_forest[2]:>10.3f}"
    f"{folds_forest[3]:>10.3f}"
    f"{folds_forest[4]:>10.3f}"
)

print(
    f"{'Extra Trees':<22}"
    f"{folds_extratrees[0]:>10.3f}"
    f"{folds_extratrees[1]:>10.3f}"
    f"{folds_extratrees[2]:>10.3f}"
    f"{folds_extratrees[3]:>10.3f}"
    f"{folds_extratrees[4]:>10.3f}"
)


print(
    f"{'Gradient Boosting':<22}"
    f"{folds_gradient[0]:>10.3f}"
    f"{folds_gradient[1]:>10.3f}"
    f"{folds_gradient[2]:>10.3f}"
    f"{folds_gradient[3]:>10.3f}"
    f"{folds_gradient[4]:>10.3f}"
)

print(
    f"{'Hist Gradient Boosting':<22}"
    f"{folds_hist_gradient_boosting[0]:>10.3f}"
    f"{folds_hist_gradient_boosting[1]:>10.3f}"
    f"{folds_hist_gradient_boosting[2]:>10.3f}"
    f"{folds_hist_gradient_boosting[3]:>10.3f}"
    f"{folds_hist_gradient_boosting[4]:>10.3f}"
)

print(
    f"{'XGBoost':<22}"
    f"{folds_xgboost[0]:>10.3f}"
    f"{folds_xgboost[1]:>10.3f}"
    f"{folds_xgboost[2]:>10.3f}"
    f"{folds_xgboost[3]:>10.3f}"
    f"{folds_xgboost[4]:>10.3f}"
)

print(
    f"{'LightGBM':<22}"
    f"{folds_lightgbm[0]:>10.3f}"
    f"{folds_lightgbm[1]:>10.3f}"
    f"{folds_lightgbm[2]:>10.3f}"
    f"{folds_lightgbm[3]:>10.3f}"
    f"{folds_lightgbm[4]:>10.3f}"
)

print(
    f"{'CatBoost':<22}"
    f"{folds_catboost[0]:>10.3f}"
    f"{folds_catboost[1]:>10.3f}"
    f"{folds_catboost[2]:>10.3f}"
    f"{folds_catboost[3]:>10.3f}"
    f"{folds_catboost[4]:>10.3f}"
)

# INFO ADICIONAL DE LASSO
print()
print("=" * 86)
print ("INFO ADICIONAL DE LASSO")
print("=" * 86)
print()
print("Features eliminadas por Lasso:", features_eliminadas_lasso)
print("FEATURES ELIMINADAS POR LASSO")
print("=" * 50)
print(pesos_lasso[pesos_lasso == 0])
print("FEATURES CONSERVADAS POR LASSO")
print("=" * 50)
print(pesos_lasso[pesos_lasso != 0])

# INFO ADICIONAL DE ELASTIC NET
print()
print("=" * 86)
print("INFO ADICIONAL DE ELASTIC NET")
print("=" * 86)
print()
print("Features eliminadas por ElasticNet:", features_eliminadas_elasticnet)
print("FEATURES ELIMINADAS POR ELASTICNET")
print("=" * 50)
print(pesos_elasticnet[pesos_elasticnet == 0])
print("FEATURES CONSERVADAS POR ELASTICNET")
print("=" * 50)
print(pesos_elasticnet[pesos_elasticnet != 0])


# INFO ADICIONAL DE DECISION TREE
print()
print("=" * 86)
print("INFO ADICIONAL DE DECISION TREE")
print("=" * 86)
print()
print("Profundidad del árbol:", profundidad_tree)
print("Número de hojas:", hojas_tree)
hojas_asignadas = modelo_tree.apply(X)
juegos_por_hoja = pd.Series(hojas_asignadas).value_counts()
print("Número de hojas:", len(juegos_por_hoja))
print("Media de juegos por hoja:", juegos_por_hoja.mean())
print("Mediana de juegos por hoja:", juegos_por_hoja.median())
print("Mínimo de juegos por hoja:", juegos_por_hoja.min())
print("Máximo de juegos por hoja:", juegos_por_hoja.max())
print()
print("Hojas con un solo juego:", (juegos_por_hoja == 1).sum())
print("MAE entrenamiento:", round(mae_train_tree, 3))
print("MAE Cross Validation:", round(mae_tree, 3))

# INFO ADICIONAL DE RANDOM FOREST
print()
print("=" * 86)
print("INFO ADICIONAL DE RANDOM FOREST")
print("=" * 86)
print()
print("Profundidad media de los árboles:", profundidad_media_forest)
print("Número medio de hojas:", hojas_medias_forest)
print()
print("MAE entrenamiento:", round(mae_train_forest, 3))
print("MAE Cross Validation:", round(mae_forest, 3))


# INFO ADICIONAL DE EXTRA TREES
print()
print("=" * 86)
print("INFO ADICIONAL DE EXTRA TREES")
print("=" * 86)
print()
print("Profundidad media de los árboles:", profundidad_media_extratrees)
print("Número medio de hojas:", hojas_medias_extratrees)
print()
print("MAE entrenamiento:", round(mae_train_extratrees, 3))
print("MAE Cross Validation:", round(mae_extratrees, 3))

# INFO ADICIONAL DE GRADIENT BOOSTING
print()
print("=" * 86)
print("INFO ADICIONAL DE GRADIENT BOOSTING")
print("=" * 86)
print()
print("MAE entrenamiento:", round(mae_train_gradient, 3))
print("MAE Cross Validation:",round(mae_gradient, 3))
print("Número de árboles:", len(modelo_gradient.estimators_))
print("Profundidad media:",sum(profundidades_gradient) / len(profundidades_gradient))
print("Número medio de hojas:",sum(hojas_gradient) / len(hojas_gradient))
print("Profundidad mínima:",min(profundidades_gradient))
print("Profundidad máxima:",max(profundidades_gradient))
print("Mínimo de hojas:",min(hojas_gradient))
print("Máximo de hojas:",max(hojas_gradient))

# INFO ADICIONAL DE HIST GRADIENT BOOSTING
print()
print("=" * 86)
print("INFO ADICIONAL DE HIST GRADIENT BOOSTING")
print("=" * 86)
print()
print("MAE entrenamiento:", round(mae_train_hist_gradient_boosting, 3))
print("MAE Cross Validation:", round(mae_hist_gradient_boosting, 3))
print("Número de iteraciones:", numero_iteraciones_hist_gradient_boosting)
print("Número total de árboles:", numero_arboles_hist_gradient_boosting)
print("Profundidad media:", round(profundidad_media_hist_gradient_boosting, 2))
print("Profundidad mínima:", profundidad_minima_hist_gradient_boosting)
print("Profundidad máxima:", profundidad_maxima_hist_gradient_boosting)
print("Número medio de hojas:", round(hojas_medias_hist_gradient_boosting, 2))
print("Mínimo de hojas:", hojas_minimas_hist_gradient_boosting)
print("Máximo de hojas:", hojas_maximas_hist_gradient_boosting)

# INFO ADICIONAL DE XGBOOST
print()
print("=" * 86)
print("INFO ADICIONAL DE XGBOOST")
print("=" * 86)
print()
print("MAE entrenamiento:",round(mae_train_xgboost, 3))
print("MAE Cross Validation:",round(mae_xgboost, 3))
print("Número total de árboles:",numero_arboles_xgboost)
print("Profundidad media:",round(profundidad_media_xgboost, 2))
print("Profundidad mínima:",profundidad_minima_xgboost)
print("Profundidad máxima:",profundidad_maxima_xgboost)
print("Número medio de hojas:",round(hojas_medias_xgboost, 2))
print("Mínimo de hojas:", hojas_minimas_xgboost)
print("Máximo de hojas:",hojas_maximas_xgboost)

# INFO ADICIONAL DE LIGHTGBM
print()
print("=" * 86)
print("INFO ADICIONAL DE LIGHTGBM")
print("=" * 86)
print()
print("MAE entrenamiento:",round(mae_train_lightgbm, 3))
print("MAE Cross Validation:",round(mae_lightgbm, 3))
print("Número total de árboles:",numero_arboles_lightgbm)
print("Profundidad media:",round(profundidad_media_lightgbm, 2))
print("Profundidad mínima:",profundidad_minima_lightgbm)
print("Profundidad máxima:",profundidad_maxima_lightgbm)
print("Número medio de hojas:",round(hojas_medias_lightgbm, 2))
print("Mínimo de hojas:",hojas_minimas_lightgbm)
print("Máximo de hojas:",hojas_maximas_lightgbm)

#INFO ADICIONAL DE CATBOOST
print()
print("=" * 86)
print("INFO ADICIONAL DE CATBOOST")
print("=" * 86)
print()
print("MAE entrenamiento:",round(mae_train_catboost, 3))
print("MAE Cross Validation:",round(mae_catboost, 3))
print("Número total de árboles:",numero_arboles_catboost)
print("Profundidad de los árboles:",profundidad_catboost)
print("Número teórico de hojas por árbol:",hojas_teoricas_catboost)




COMPARACIÓN DEL RENDIMIENTO DE LOS MODELOS
MODELO                       MAE     MEJORA VS DUMMY      INTERCEPTO
------------------------------------------------------------------------------
Dummy (media)              1.651              +0.000               -
Linear Regression          1.703              -0.052           5.258
Ridge                      1.427              +0.223           6.977
Lasso                      1.257              +0.394           6.977
Elastic Net                1.236              +0.415           6.977
Decission Tree             1.632              +0.019               -
Random Forest              1.275              +0.375               -
Extra Trees                1.312              +0.338               -
Gradient Boosting          1.261              +0.390               -
Hist Gradient Boosting     1.427              +0.223               -
XGBoost                    1.353              +0.297               -
LightGBM                   1.415              +0.

# **PASO 10: PREPARAMOS EL DATA FRAME DE JUEGOS QUE VAMOS A EVALUAR CON LOS MODELOS QUE HAN "APRENDIDO" EN LOS PASOS ANTERIORES**

In [25]:
# ==============================================================
# PREPARACION DEL DATA FRAME DE JUEGOS COMPLETO
# ==============================================================
#
# ACLARACION IMPORTANTE: DISTINCION ENTRE df_train y df_games_to_evaluate
#
# df_train
# --------
# Contiene los juegos que YA he puntuado.
#
# Esos juegos los he puntuado en la BGG y ya los he utilizado
# para ENTRENAR los modelos.
#
# En otras palabras:
#     df_train responde a:
#     "¿CON QUÉ DATOS APRENDE EL MODELO?"
#
# df_games_to_evaluate
# --------------------
# Contiene los juegos que NO he puntuado todavía.
#
# Sobre estos juegos no conocemos UserRating.
# Por tanto, NO sirven para entrenar.
#
# Se utilizan después del entrenamiento para que los modelos
# generen una predicción mediante predict().
#
# En otras palabras:
#
#     df_games_to_evaluate responde a:
#     "¿SOBRE QUÉ JUEGOS QUIERO QUE EL MODELO PREDIGA?"
#
#
# El flujo general es:
#
#     Juegos puntuados
#          ↓
#       df_train
#          ↓
#      entrenamiento
#          ↓
#        modelo
#
#
#     Juegos NO puntuados
#          ↓
#  df_games_to_evaluate
#          ↓
#       predict()
#          ↓
#  df_evaluation_results
#
#
# IMPORTANTE:
# df_train y df_games_to_evaluate pueden tener las mismas columnas
# descriptivas/features, pero representan conjuntos de juegos
# distintos y tienen funciones completamente diferentes dentro
# del proyecto.
# ==============================================================
# PREPARACION DE df_games_to_evaluate
#
# Partimos del conjunto global de juegos.
#
# Eliminamos los juegos que ya hemos puntuado porque esos juegos
# ya han participado en el entrenamiento de los modelos.
#
# df_games_to_evaluate contiene, por tanto, todos los juegos
# sobre los que posteriormente queremos generar predicciones.
# ==============================================================

df_games_to_evaluate = df_games_encoded[
    ~df_games_encoded['BGGId'].isin(df_rated['BGGId'])
].copy()

# ==============================================================
# TRATAMIENTO DE VALORES AUSENTES
# ==============================================================
#
# Durante el entrenamiento sustituimos los valores ausentes de
# ComAgeRec y LanguageEase por la mediana calculada en df_train.
#
# Para mantener exactamente el mismo tratamiento de datos,
# hacemos lo mismo ahora con los juegos que queremos evaluar.
#
# IMPORTANTE:
# utilizamos la mediana de df_train, NO la mediana de
# df_games_to_evaluate.
# ==============================================================

df_games_to_evaluate['ComAgeRec'] = (
    df_games_to_evaluate['ComAgeRec']
    .fillna(df_train['ComAgeRec'].median())
)

df_games_to_evaluate['LanguageEase'] = (
    df_games_to_evaluate['LanguageEase']
    .fillna(df_train['LanguageEase'].median())
)


df_games_to_evaluate_mechanics = df_games_to_evaluate.merge(
    df_mechanics,
    on='BGGId',
    how='left'
)

df_games_to_evaluate_themes = df_games_to_evaluate_mechanics.merge(
    df_themes,
    on='BGGId',
    how='left'
)

df_games_to_evaluate_subcategories = df_games_to_evaluate_themes.merge(
    df_subcategories,
    on='BGGId',
    how='left'
)

df_games_to_evaluate_designers = df_games_to_evaluate_subcategories.merge(
    df_designers,
    on='BGGId',
    how='left'
)

df_games_to_evaluate_publishers = df_games_to_evaluate_designers.merge(
    df_publishers,
    on='BGGId',
    how='left'
)

print("BASE")
print(df_train.shape)
print(df_games_to_evaluate.shape)

print("\nMECHANICS")
print(df_train_mechanics.shape)
print(df_games_to_evaluate_mechanics.shape)

print("\nTHEMES")
print(df_train_themes.shape)
print(df_games_to_evaluate_themes.shape)

print("\nSUBCATEGORIES")
print(df_train_subcategories.shape)
print(df_games_to_evaluate_subcategories.shape)

print("\nDESIGNERS")
print(df_train_designers.shape)
print(df_games_to_evaluate_designers.shape)

print("\nPUBLISHERS")
print(df_train_publishers.shape)
print(df_games_to_evaluate_publishers.shape)





BASE
(259, 1512)
(21666, 1511)

MECHANICS
(259, 1669)
(21666, 1668)

THEMES
(259, 1886)
(21666, 1885)

SUBCATEGORIES
(259, 1896)
(21666, 1895)

DESIGNERS
(259, 3489)
(21666, 3488)

PUBLISHERS
(259, 5354)
(21666, 5353)


# PASO 11: HACEMOS LAS PREDICCIONES CON LOS MODELOS ENTRENADOS

In [53]:
#Creamos la matriz de features que recibirán los modelos.
#
# Debe contener EXACTAMENTE las mismas columnas
# utilizadas durante el entrenamiento.

#X_to_evaluate = df_games_to_evaluate[features_games]
#X_to_evaluate = df_games_to_evaluate_mechanics[features_games_mechanics]
#X_to_evaluate = df_games_to_evaluate_themes[features_games_themes]
#X_to_evaluate = df_games_to_evaluate_subcategories[features_games_subcategories]
#X_to_evaluate = df_games_to_evaluate_designers[features_games_designers]
X_to_evaluate = df_games_to_evaluate_publishers[features_games_publishers]

# Diccionario de modelos ya entrenados
modelos = {
    'Linear': modelo_linear,
    'Ridge': modelo_ridge,
#   'Huber': modelo_huber,
    'Lasso': modelo_lasso,
    'ElasticNet': modelo_elasticnet,
    'DecisionTree': modelo_tree,
    'RandomForest': modelo_forest,
    'ExtraTrees': modelo_extratrees,
    'GradientBoosting': modelo_gradient,
    'HistGradientBoosting': modelo_hist_gradient_boosting,
    'XGBoost': modelo_xgboost,
    'LightGBM': modelo_lightgbm,
    'CatBoost': modelo_catboost
}


# Creamos el dataframe donde guardaremos los resultados.
#
# Partimos de df_games_to_evaluate para conservar
# BGGId, Name y demás información de cada juego.

df_evaluation_results = df_games_to_evaluate.copy()

# Antes de evaluar los juegos, incluimos una columna que nos dice si poseemos el juego aunquue no lo hayamos puntuado
df_evaluation_results['Own'] = (
    df_evaluation_results['BGGId']
    .isin(df_owned['BGGId'])
    .astype(int)
)

# Ejecutamos cada modelo
for nombre_modelo, modelo in modelos.items():

    if nombre_modelo == 'Huber':
        continue

    columna_pred = 'Pred_' + nombre_modelo
    columna_rank = 'Rank_' + nombre_modelo


    # ============================================================
    # PREPARACIÓN DE LOS DATOS PARA EL MODELO
    # ============================================================

    # Cada modelo recibe una versión de X_to_evaluate
    # preparada de acuerdo con sus necesidades.
    #
    # Para la mayoría de modelos no habrá cambios.
    # Para XGBoost, LightGBM, CatBoost, etc.,
    # la función aplicará las transformaciones necesarias.

    X_para_predecir = prepara_X_para_modelo(
        X_to_evaluate,
        nombre_modelo
    )


    # ============================================================
    # PREDICCIÓN
    # ============================================================

    df_evaluation_results[columna_pred] = (
        modelo.predict(
            X_para_predecir
        )
    )


    # ============================================================
    # RANKING
    # ============================================================

    df_evaluation_results[columna_rank] = (
        df_evaluation_results[columna_pred]
        .rank(
            ascending=False,
            method='min'
        )
        .astype(int)
    )

In [55]:
# ==============================================================
# TOP 100 DE CADA MODELO
# ==============================================================

for nombre_modelo in modelos:

    if nombre_modelo == 'Huber':
        continue
    columna_pred = 'Pred_' + nombre_modelo
    columna_rank = 'Rank_' + nombre_modelo

    print()
    print("=" * 90)
    print("TOP 100 -", nombre_modelo.upper())
    print("=" * 90)

    display(
        df_evaluation_results[
            [
                'BGGId',
                'Name',
                'Own',
                columna_pred,
                columna_rank
            ]
        ]
        .sort_values(columna_rank)
        .head(100)
        .style.format({
            columna_pred: '{:.2f}'
        })
    )



TOP 100 - LINEAR


,BGGId,Name,Own,Pred_Linear,Rank_Linear
3025,4815,The Campaign for North Africa: The Desert War 1940-43,0,263.26,1
7808,29285,Case Blue,0,99.39,2
9259,46669,1914: Offensive à outrance,0,79.58,3
13495,158793,Atlantic Wall: D-Day to Falaise,0,65.74,4
3909,6942,Drang Nach Osten!,0,58.49,5
217,254,Empires in Arms,0,56.00,6
18186,239982,The Enigma Box,0,46.31,7
18051,236650,1985: Under an Iron Sky,0,42.97,8
16636,209511,Atlanta Is Ours,0,38.86,9
3384,5622,Pacific War: The Struggle Against Japan 1941-1945,0,33.37,10



TOP 100 - RIDGE


,BGGId,Name,Own,Pred_Ridge,Rank_Ridge
3025,4815,The Campaign for North Africa: The Desert War 1940-43,0,226.61,1
9259,46669,1914: Offensive à outrance,0,70.71,2
7808,29285,Case Blue,0,66.61,3
3909,6942,Drang Nach Osten!,0,51.36,4
13495,158793,Atlantic Wall: D-Day to Falaise,0,43.84,5
18186,239982,The Enigma Box,0,38.49,6
217,254,Empires in Arms,0,38.27,7
18051,236650,1985: Under an Iron Sky,0,29.91,8
16636,209511,Atlanta Is Ours,0,27.57,9
3394,5651,The Longest Day,0,26.20,10



TOP 100 - LASSO


...

,BGGId,Name,Own,Pred_XGBoost,Rank_XGBoost
3619,6202,The Rise of the Roman Republic,0,9.88,1
11765,129122,Band of Brothers: Ghost Panzer,0,9.74,2
6989,22377,Spartacus,0,9.71,3
6245,17393,Pax Romana,0,9.56,4
3215,5233,The Great Battles of Alexander,0,9.49,5
12544,144189,Fire in the Lake,1,9.48,6
9498,58624,Storm Over Dien Bien Phu,0,9.48,7
14829,179251,Urban Operations,0,9.48,8
4273,8037,Patrol!: Man-to-Man Combat in the 20th Century,0,9.47,9
4908,10022,La Révolution française: La patrie en danger 1791-1795,0,9.46,10



TOP 100 - LIGHTGBM


,BGGId,Name,Own,Pred_LightGBM,Rank_LightGBM
8106,32327,Waterloo 1815,0,10.15,1
9791,67084,The War: Europe 1939-1945,0,9.99,2
8410,35476,Barbarossa: Crimea,0,9.94,3
3851,6826,The Campaigns of Robert E. Lee,0,9.93,4
20164,280106,Britannia: Classic and Duel Edition,0,9.91,5
9259,46669,1914: Offensive à outrance,0,9.90,6
2118,3050,"Napoleon at Bay: Defend the Gates of Paris, 1814",0,9.90,7
9564,61487,Unconditional Surrender! World War 2 in Europe,1,9.88,8
4084,7502,Leros,0,9.87,9
3071,4921,"3rd Fleet: Modern Naval Combat in the North Pacific, Caribbean, and Atlantic Oceans",0,9.87,10



TOP 100 - CATBOOST


,BGGId,Name,Own,Pred_CatBoost,Rank_CatBoost
12544,144189,Fire in the Lake,1,9.52,1
13948,165872,Liberty or Death: The American Insurrection,1,9.50,2
6245,17393,Pax Romana,0,9.38,3
10465,91080,Andean Abyss,1,9.37,4
5763,13855,Carthage: The First Punic War,0,9.36,5
3619,6202,The Rise of the Roman Republic,0,9.35,6
16278,203624,Mea Culpa,0,9.31,7
3879,6896,MechWar 2: Red Star / White Star,0,9.31,8
7506,26457,Successors (Third Edition),0,9.31,9
3215,5233,The Great Battles of Alexander,0,9.30,10
